<a href="https://colab.research.google.com/github/LujainAlawwad/ImplicitAspectExtraction/blob/main/6.Arabic_Knowledge_Graph_(CrossLingual_VS_Arabic_KG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic KG Inference and Evaluation
## Two strategies:
1. **Strategy 1**: Cross-lingual English KG — multilingual embeddings map Arabic adjectives to English KG nodes
2. **Strategy 2**: Arabic domain-specific KG — direct lexical lookup with Arabic clue matching

### Datasets:
- **M-ABSA Arabic**: `/content/drive/MyDrive/colab_files/m-absa-arabic-test.csv` → gold `category` in `ENTITY#ASPECT` format (English) → Strategy 1 only
- **SemEval2016**: in `Arabic_test2.csv` → gold `category` in `ENTITY#ASPECT` format (English) → Strategy 1 only
- **HAAD**: in `Arabic_test2.csv` → gold `category` is Arabic aspect-only (no entity) → Strategy 2 only


## Block 0 — Installations

In [ ]:
# Run once per environment
!pip -q install networkx sentence-transformers pandas tqdm

try:
    import stanza
except ImportError:
    !pip -q install stanza

try:
    from camel_tools.tokenizers.word import simple_word_tokenize
except ImportError:
    !pip install -q camel-tools
    !camel_data -i morphology-db-msa-r13
    !camel_data -i disambig-mle-calima-msa-r13

try:
    import stanza
    stanza.download('ar')
except Exception:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.6 MB/s eta 0:00:00
   ━━

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: ar (Arabic) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ar/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


## Block 1 — Imports

In [ ]:
from google.colab import drive


drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import json, re, ast, os
import numpy as np
import pandas as pd
import stanza
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.disambig.mle import MLEDisambiguator
tqdm.pandas()
print("✅ Imports done.")


✅ Imports done.


## Block 2 — Load Models (run once per session)

In [ ]:
print("Loading CAMeL MLE disambiguator...")
CAMEL_MLE = MLEDisambiguator.pretrained()
print("✅ CAMeL MLE ready.")

print("\nLoading Stanza Arabic pipeline...")
ar_nlp = stanza.Pipeline(
    lang='ar',
    processors='tokenize,pos,lemma,depparse',
    tokenize_no_ssplit=True,
    use_gpu=True,
    verbose=False,
)
print("✅ Stanza Arabic pipeline ready.")

print("\nLoading multilingual sentence encoder...")
encoder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ Encoder ready.")


Loading CAMeL MLE disambiguator...
✅ CAMeL MLE ready.

Loading Stanza Arabic pipeline...
✅ Stanza Arabic pipeline ready.

Loading multilingual sentence encoder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Encoder ready.


## Block 3 — Paths

In [ ]:
# ── KG files ─────────────────────────────────────────────────────
ENG_KG_PATH     = "/content/drive/MyDrive/colab_files/Knowledge_graph_mabsaEnriched.json"
ENG_KG_ENCODED  = "/content/drive/MyDrive/colab_files/encoded_kg/encoded_kg_minilm.pkl"
AR_KG_PATH      = "/content/drive/MyDrive/colab_files/arabic_enriched_kg.json"

# ── Dataset files ─────────────────────────────────────────────────
MABSA_AR_PATH   = "/content/drive/MyDrive/colab_files/m-absa-arabic-test.csv"
ARABIC_TEST2    = "/content/drive/MyDrive/colab_files/Arabic_test2.csv"
# Arabic_test2.csv contains both SemEval2016 and HAAD splits
# separated by 'dataset' or 'source_dataset' column

# ── Output files ──────────────────────────────────────────────────
OUT_MABSA_AR    = "KG_Strategy1_MABSA_AR_predictions.csv"
OUT_SE16_AR     = "KG_Strategy1_SemEval16_AR_predictions.csv"
OUT_HAAD_AR     = "KG_Strategy2_HAAD_predictions.csv"


## Block 4 — Arabic NLP Helpers (shared by both strategies)

In [ ]:
# ================================================================
# Arabic normalisation, CAMeL lemmatisation, lonely adjective
# detection — shared by Strategy 1 and Strategy 2
# ================================================================

_DIAC = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def norm_ar(s: str) -> str:
    """Full Arabic surface normalisation."""
    if not isinstance(s, str):
        return ''
    t = _DIAC.sub('', s).replace('\u0640', '')
    t = (t.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا')
          .replace('ؤ', 'و').replace('ئ', 'ي')
          .replace('ى', 'ي').replace('ة', 'ه'))
    if t.startswith('ال') and len(t) > 2:
        t = t[2:]
    return ' '.join(t.split())


def _camel_lemma(surface: str) -> str:
    """Get best CAMeL morphological lemma for a surface word."""
    try:
        words    = simple_word_tokenize(surface)
        disambig = CAMEL_MLE.disambiguate(words)
        if disambig and disambig[0].analyses:
            return disambig[0].analyses[0].analysis.get('lex', surface)
    except Exception:
        pass
    return surface


# ================================================================
# FIX 1: Feminine adjective masculine base reduction
# Strips gender/number/case suffixes to recover masculine singular
# base form for KG lookup. Uses CAMeL POS features as guard for
# the ه suffix (feminine singular) to avoid stripping root-final ة.
# ================================================================

_AR_ADJ_SUFFIXES_ORDERED = [
    'ين',   # masculine plural / dual oblique
    'ون',   # masculine plural nominative
    'ات',   # feminine plural
    'ان',   # dual nominative
    'ه',    # feminine singular (ة → ه after norm_ar)
    'ا',    # accusative tanwin (ًا → ا after norm_ar)
]
_MIN_ROOT_LEN = 3


def _camel_features(surface: str) -> dict:
    """Return CAMeL morphological analysis dict for a surface word."""
    try:
        tokens   = simple_word_tokenize(surface)
        disambig = CAMEL_MLE.disambiguate(tokens)
        if disambig and disambig[0].analyses:
            return disambig[0].analyses[0].analysis
    except Exception:
        pass
    return {}


def _is_feminine_adj(surface: str, features: dict) -> bool:
    """
    Returns True if CAMeL confirms this token is a feminine adjective.
    Primary signal : BW tag contains NSUFF_FEM_SG (unambiguous suffix)
    Secondary signal: pos == 'adj' (catches cases where gen=m but IS adj)
    Excludes pos == 'noun' to protect حياة, قناة, ساعة etc.
    """
    if not features:
        return False
    bw  = features.get('bw', '')
    pos = features.get('pos', '')
    if 'NSUFF_FEM_SG' in bw:
        return True
    if str(pos).lower() == 'adj':
        return True
    return False


def reduce_to_masculine_base(norm_form: str, surface: str = '') -> str:
    """
    Strip Arabic adjective inflection suffixes after norm_ar() to
    recover the masculine singular base for KG lookup.
    The ه suffix (feminine singular) requires CAMeL confirmation
    to avoid stripping root-final ة in nouns.
    All other suffixes (plural, dual, tanwin) are stripped directly.
    """
    candidate_suffix = None
    candidate_base   = None
    for suffix in _AR_ADJ_SUFFIXES_ORDERED:
        if norm_form.endswith(suffix):
            base = norm_form[: -len(suffix)]
            if len(base) >= _MIN_ROOT_LEN:
                candidate_suffix = suffix
                candidate_base   = base
                break

    if candidate_base is None:
        return norm_form

    if candidate_suffix == 'ه':
        features = _camel_features(surface) if surface else {}
        if not _is_feminine_adj(surface, features):
            return norm_form

    return candidate_base


# ── Lonely adjective detection ────────────────────────────────────

GENERIC_SENT_AR = {
    'ممتاز','رائع','جيد','جيده','حلو','حلوه','كويس','كويسه',
    'تمام','عظيم','مذهل','سيء','سئ','وحش','ردي','عادي',
    'مقبول','لطيف','قوي','ضعيف','مميز','مثالي','جميل',
}

ADJ_SURFACE_BLOCKLIST = {
    'بعيداً', 'بعيدا', 'قريباً', 'قريبا',
    'سريعاً', 'سريعا', 'كثيراً', 'كثيرا',
    'مباشرة', 'تلقائياً', 'تلقائيا',
    'كثيرًا','كثيرا','قليلًا','قليلا','دائمًا','دائما',
    'أحيانًا','أحيانا','نادرًا','نادرا','حاليًا','حاليا',
    'أخيرًا','أخيرا','لاحقًا','لاحقا','سابقًا','سابقا',
    'حالياً','مؤخرًا','مؤخرا','مسبقًا','مسبقا',
    'جدًا','جدا','تمامًا','تماما','فعلًا','فعلا',
    'مطلقًا','مطلقا','أبدًا','أبدا','بالفعل',
    'أولاً','اولاً','أولا','اولا','ثانيًا','ثانياً','ثانيا',
    'ثالثًا','ثالثاً','ثالثا','رابعًا','رابعاً','رابعا',
    'شخصياً','شخصيا',
    'واحدة','واحد','اثنان','اثنتان','كلا','كلتا',
    'بعض','معظم','كل','جميع','أي','كثير','اكثر','أكثر',
    'اقل','أقل','اول','أول',
    'اصغر','أصغر','اكبر','أكبر','افضل','أفضل','اسوا','أسوأ',
    'مفردي','منفردًا','منفردا',
    'الهندي','الصيني','الإيطالي','العربي','الأمريكي',
    'الفرنسي','التركي','المغربي','اللبناني',
    'يومي','يوميا','يومياً','اسبوعي','شهري','سنوي',
    'اخير','الاخير','اللازم','لازم','ثاني','اخر','آخر',
}

NON_OPINION_DEP_RELS = {
    'nummod', 'det', 'advmod', 'compound', 'fixed', 'flat',
}

# FIX 2: Only these deprels on noun children indicate the adjective
# is genuinely modifying an aspect noun (should block lonely detection).
# nsubj (subject) and nmod (compound unit) do NOT block.
BLOCKING_NOUN_DEPRELS = {
    'obj',
    'obl:arg',
}

# Adjective-like suffixes in Arabic (morphological heuristic)
_ADJ_SUFFIXES = ('ة', 'ه', 'ي', 'ية', 'يه', 'ين', 'ان', 'ات')

# Known sentiment words that Stanza often mislabels as NOUN or ADV
AR_SENTIMENT_NOUNS = {
    # Generic sentiment
    'رائع','رايع','ممتاز','جميل','مذهل','عظيم','جيد','حلو','كويس',
    'سيء','سئ','وحش','ردي','ضعيف','مقبول','عادي','تمام',
    # Quality descriptors
    'لذيذ','طازج','طري','مقرمش','ناعم','صلب','ثقيل','خفيف',
    'مريح','هادي','هادئ','نظيف','وسخ','متسخ',
    # Hotel specific
    'فخم','راقي','انيق','واسع','ضيق','قديم','جديد','حديث',
    # Exclamation fragments
    'رائعة','ممتازة','جميلة','مذهلة','عظيمة','رهيب','رهيبة',
    'مثالي','مثالية','مميز','مميزة',
}


def find_lonely_adjectives_ar(sentence: str) -> List[Dict]:
    """
    Detect lonely adjectives in Arabic.

    FIX 1: masculine base reduction added to adj_info for KG lookup.
    FIX 2: compound adjective-noun detection (nmod children).
           Only BLOCKING_NOUN_DEPRELS (obj, obl:arg) block detection.
           nsubj children (subject nouns) do NOT block.
           nmod children are treated as compound adjectival units.
    """
    if not sentence or not isinstance(sentence, str):
        return []
    try:
        doc = ar_nlp(sentence)
    except Exception:
        return []

    lonely = []
    for sent in doc.sentences:
        tokens = sent.words
        for tok in tokens:
            is_adj = tok.upos == 'ADJ'
            is_adj_like_noun = (
                tok.upos == 'NOUN' and
                tok.deprel not in ('nsubj', 'obj', 'obl',
                                   'nmod', 'iobj', 'compound') and
                (norm_ar(tok.text) in {norm_ar(w) for w in AR_SENTIMENT_NOUNS} or
                 any(tok.text.endswith(suf) for suf in _ADJ_SUFFIXES))
            )
            if not (is_adj or is_adj_like_noun):
                continue

            if tok.text in ADJ_SURFACE_BLOCKLIST:
                continue
            if norm_ar(tok.text) in {norm_ar(b) for b in ADJ_SURFACE_BLOCKLIST}:
                continue
            if tok.deprel in NON_OPINION_DEP_RELS:
                continue

            head = tokens[tok.head - 1] if tok.head > 0 else None

            if is_adj:
                if head and head.upos in ('NOUN', 'PROPN', 'X', 'NUM', 'DET'):
                    continue
            else:  # adj_like_noun
                if tok.deprel not in ('root', 'nsubj', 'csubj', 'xcomp',
                                      'appos', 'conj'):
                    continue
                if head and head.upos in ('NOUN', 'PROPN'):
                    continue

            children = [t for t in tokens if t.head == tok.id]

            # FIX 2: nmod noun children → compound adjectival unit
            nmod_nouns = [c for c in children
                          if c.upos in ('NOUN', 'PROPN')
                          and c.deprel == 'nmod']

            # Only block on true aspect-modifying deprels
            blocking_nouns = [c for c in children
                               if c.upos in ('NOUN', 'PROPN', 'X')
                               and c.deprel in BLOCKING_NOUN_DEPRELS]

            if blocking_nouns:
                continue

            camel_lem = _camel_lemma(tok.text)
            is_generic = (
                norm_ar(tok.text) in GENERIC_SENT_AR
                or norm_ar(camel_lem) in GENERIC_SENT_AR
            )

            # FIX 1: compute masculine base for KG lookup
            norm_surface  = norm_ar(tok.text)
            norm_camel    = norm_ar(camel_lem)
            masc_base     = reduce_to_masculine_base(norm_surface,
                                                     surface=tok.text)
            masc_base_cam = reduce_to_masculine_base(norm_camel,
                                                     surface=camel_lem)

            compound_noun = nmod_nouns[0].text if nmod_nouns else None

            lonely.append({
                'surface':        tok.text,
                'lemma_stanza':   tok.lemma or tok.text,
                'lemma_camel':    camel_lem,
                'norm_ar':        norm_camel,
                'masc_base':      masc_base,       # FIX 1
                'masc_base_cam':  masc_base_cam,   # FIX 1
                'dep_rel':        tok.deprel,
                'head_text':      head.text if head else 'ROOT',
                'head_upos':      head.upos if head else 'ROOT',
                'is_generic':     is_generic,
                'compound_noun':  compound_noun,   # FIX 2
                'is_compound':    compound_noun is not None,  # FIX 2
            })
    return lonely


# ── Domain inference (centroid method, multilingual encoder) ───────

DOMAIN_VOCAB = {
    'restaurant': [
        'food', 'meal', 'restaurant', 'menu', 'waiter', 'service',
        'dish', 'delicious', 'portion', 'taste', 'chef', 'cuisine',
        'dining', 'reservation', 'appetizer', 'dessert', 'drink',
        'fresh', 'flavor', 'ambience', 'table', 'order', 'cook',
    ],
    'food': [
        'food', 'ingredient', 'recipe', 'grocery', 'product', 'flavor',
        'taste', 'package', 'nutrition', 'diet', 'snack', 'meal',
        'fresh', 'organic', 'delivery', 'brand', 'quality', 'price',
    ],
    'hotel': [
        'hotel', 'room', 'check-in', 'checkout', 'staff', 'reception',
        'bed', 'bathroom', 'pool', 'lobby', 'breakfast', 'housekeeping',
        'accommodation', 'stay', 'booking', 'amenity', 'facility',
        'clean', 'comfortable', 'view', 'floor', 'elevator', 'towel',
    ],
    'laptop': [
        'laptop', 'computer', 'keyboard', 'screen', 'battery', 'processor',
        'RAM', 'storage', 'display', 'charger', 'performance', 'speed',
        'software', 'hardware', 'port', 'wifi', 'boot', 'weight', 'design',
    ],
    'phone': [
        'phone', 'smartphone', 'camera', 'screen', 'battery', 'charging',
        'app', 'signal', 'call', 'speaker', 'microphone', 'display',
        'RAM', 'storage', 'fingerprint', 'network', 'sim', 'bluetooth',
    ],
    'coursera': [
        'course', 'lecture', 'assignment', 'certificate', 'instructor',
        'video', 'quiz', 'learning', 'student', 'grade', 'content',
        'material', 'online', 'module', 'lesson', 'skill', 'university',
    ],
    'books': [
        'book', 'novel', 'author', 'story', 'chapter', 'plot',
        'character', 'narrative', 'reading', 'fiction', 'pages',
        'publish', 'genre', 'writing', 'reader', 'prose', 'biography',
    ],
    'sight': [
        'tourist', 'attraction', 'landmark', 'visit', 'sightseeing',
        'museum', 'monument', 'historic', 'tour', 'scenery', 'view',
        'destination', 'travel', 'heritage', 'site', 'entrance', 'ticket',
    ],
}

_domain_embeddings: Dict[str, np.ndarray] = {}


def build_domain_index(enc) -> None:
    global _domain_embeddings
    _domain_embeddings = {}
    for domain, vocab in DOMAIN_VOCAB.items():
        word_embs = enc.encode(
            vocab, normalize_embeddings=True,
            convert_to_numpy=True, show_progress_bar=False,
        )
        centroid = word_embs.mean(axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        _domain_embeddings[domain] = centroid
    print(f"✅ Domain index built: {list(_domain_embeddings.keys())}")


def infer_domain_from_sentence(sentence: str, enc) -> str:
    if not sentence or not _domain_embeddings:
        return 'unknown'
    sent_emb = enc.encode(
        sentence, normalize_embeddings=True, convert_to_numpy=True
    )
    best_domain, best_sim = 'unknown', -1.0
    for domain, centroid in _domain_embeddings.items():
        sim = float(np.dot(sent_emb, centroid))
        if sim > best_sim:
            best_sim    = sim
            best_domain = domain
    return best_domain


build_domain_index(encoder)
print("✅ Arabic NLP helpers ready (FIX1: feminine reduction, FIX2: compound detection).")


✅ Domain index built: ['restaurant', 'food', 'hotel', 'laptop', 'phone', 'coursera', 'books', 'sight']
✅ Arabic NLP helpers ready (FIX1: feminine reduction, FIX2: compound detection).


## Block 5 — Load Datasets

In [ ]:
# def safe_literal_eval(x):
#     if isinstance(x, list): return x
#     if x is None or (isinstance(x, float) and pd.isna(x)): return []
#     try:
#         return ast.literal_eval(str(x))
#     except (ValueError, SyntaxError):
#         return [str(x)]

# def clean_list_values(x):
#     if isinstance(x, list):
#         return [str(v).strip() for v in x
#                 if v is not None
#                 and not (isinstance(v, float) and pd.isna(v))
#                 and str(v).strip().lower() not in ('nan', 'none', '')]
#     if x is None or (isinstance(x, float) and pd.isna(x)):
#         return []
#     return [str(x).strip()]

# def dedup_gold_field(val) -> list:
#     """Remove exact duplicate gold labels preserving order."""
#     if isinstance(val, list):
#         items = [str(v).strip() for v in val]
#     else:
#         s = str(val).strip()
#         if s.startswith('['):
#             try:
#                 parsed = ast.literal_eval(s)
#                 items = [str(v).strip() for v in parsed]
#             except Exception:
#                 items = [s]
#         else:
#             items = [s]
#     seen, deduped = set(), []
#     for item in items:
#         key = item.strip().lower()
#         if key and key not in seen:
#             seen.add(key)
#             deduped.append(item)
#     return deduped if deduped else ['none']


# # ── M-ABSA Arabic ─────────────────────────────────────────────────
# mabsa_ar = pd.read_csv(MABSA_AR_PATH)
# mabsa_ar['aspect']   = mabsa_ar['aspect'].apply(safe_literal_eval).apply(clean_list_values)
# mabsa_ar['category'] = mabsa_ar['category'].apply(dedup_gold_field)
# if 'domain' not in mabsa_ar.columns:
#     mabsa_ar['domain'] = 'unknown'
# mabsa_ar.reset_index(drop=True, inplace=True)
# print(f"✅ M-ABSA Arabic: {len(mabsa_ar)} rows")
# print(f"   Types:   {dict(mabsa_ar['type'].value_counts()) if 'type' in mabsa_ar.columns else 'N/A'}")
# print(f"   Domains: {dict(mabsa_ar['domain'].value_counts())}")

# # ── Arabic_test2: SemEval2016 + HAAD ──────────────────────────────
# arabic2 = pd.read_csv(ARABIC_TEST2)
# arabic2['aspect']   = arabic2['aspect'].apply(safe_literal_eval).apply(clean_list_values)
# arabic2['category'] = arabic2['category'].apply(dedup_gold_field)
# if 'domain' not in arabic2.columns:
#     arabic2['domain'] = 'unknown'
# arabic2.reset_index(drop=True, inplace=True)
# print(f"\n✅ Arabic_test2: {len(arabic2)} rows")

# # Identify split column
# split_col = next((c for c in ['dataset','source_dataset'] if c in arabic2.columns), None)
# if split_col:
#     print(f"   Split on '{split_col}': {dict(arabic2[split_col].value_counts())}")
# else:
#     print("   ⚠ No 'dataset' or 'source_dataset' column found — check file")
#     split_col = arabic2.columns[0]

# print(f"   Types: {dict(arabic2['type'].value_counts()) if 'type' in arabic2.columns else 'N/A'}")

# # ── Split into SemEval2016 and HAAD ───────────────────────────────
# se16_ar = arabic2[
#     arabic2[split_col].str.lower().str.contains('semeval|se16|2016', na=False)
# ].copy().reset_index(drop=True)

# haad_ar = arabic2[
#     arabic2[split_col].str.lower().str.contains('haad', na=False)
# ].copy().reset_index(drop=True)

# if len(se16_ar) == 0 and len(haad_ar) == 0:
#     print(f"\n⚠ Auto-split failed. Unique values in '{split_col}':")
#     print(arabic2[split_col].unique())
#     print("Update the str.contains() patterns above to match your values.")
# else:
#     print(f"\n   SemEval16 Arabic : {len(se16_ar)} rows")
#     print(f"   HAAD             : {len(haad_ar)} rows")


In [ ]:
# ================================================================
# BLOCK 5 — Load data from composite predictions file
# Uses composite_predictions_all_datasets_ar.csv as single source
# to guarantee sentence key alignment with KG inference output.
# ================================================================

import pandas as pd, ast

COMPOSITE_PATH = "composite_predictions_all_datasets_ar.csv"

def safe_literal_eval(x):
    if isinstance(x, list): return x
    if x is None or (isinstance(x, float) and pd.isna(x)): return []
    try:
        return ast.literal_eval(str(x))
    except (ValueError, SyntaxError):
        return [str(x)]

def clean_list_values(x):
    if isinstance(x, list):
        return [str(v).strip() for v in x
                if v is not None
                and not (isinstance(v, float) and pd.isna(v))
                and str(v).strip().lower() not in ('nan', 'none', '')]
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    return [str(x).strip()]

def dedup_gold_field(val) -> list:
    if isinstance(val, list):
        items = [str(v).strip() for v in val]
    else:
        s = str(val).strip()
        if s.startswith('['):
            try:
                parsed = ast.literal_eval(s)
                items = [str(v).strip() for v in parsed]
            except Exception:
                items = [s]
        else:
            items = [s]
    seen, deduped = set(), []
    for item in items:
        key = item.strip().lower()
        if key and key not in seen:
            seen.add(key)
            deduped.append(item)
    return deduped if deduped else ['none']

# ── Load full composite ──────────────────────────────────────────
composite = pd.read_csv(COMPOSITE_PATH)
composite['aspect']   = composite['aspect'].apply(safe_literal_eval).apply(clean_list_values)
composite['category'] = composite['category'].apply(dedup_gold_field)
if 'domain' not in composite.columns:
    composite['domain'] = 'unknown'
composite.reset_index(drop=True, inplace=True)

print(f"✅ Composite loaded: {len(composite)} rows")
print(f"   Datasets: {dict(composite['dataset'].value_counts())}")
print(f"   Types:    {dict(composite['type'].value_counts())}")

# ── Split by dataset ─────────────────────────────────────────────
mabsa_ar = composite[
    composite['dataset'].str.lower().str.contains('mabsa|m-absa', na=False)
].copy().reset_index(drop=True)

se16_ar = composite[
    composite['dataset'].str.lower().str.contains('semeval|se16|2016', na=False)
].copy().reset_index(drop=True)

haad_ar = composite[
    composite['dataset'].str.lower().str.contains('haad', na=False)
].copy().reset_index(drop=True)

print(f"\n   M-ABSA Arabic : {len(mabsa_ar)} rows  "
      f"types={dict(mabsa_ar['type'].value_counts())}")
print(f"   SemEval2016   : {len(se16_ar)} rows  "
      f"types={dict(se16_ar['type'].value_counts())}")
print(f"   HAAD          : {len(haad_ar)} rows  "
      f"types={dict(haad_ar['type'].value_counts())}")

✅ Composite loaded: 3707 rows
   Datasets: {'m-absa-arabic': 2182, 'SemEval2016': 1226, 'HAAD': 299}
   Types:    {'explicit': 2785, 'implicit': 814, 'mixed': 104, 'objective': 4}

   M-ABSA Arabic : 2182 rows  types={'explicit': 1356, 'implicit': 718, 'mixed': 104, 'objective': 4}
   SemEval2016   : 1226 rows  types={'explicit': 1130, 'implicit': 96}
   HAAD          : 299 rows  types={'explicit': 299}


In [ ]:
# Add this at the end of Block 5, after loading composite:
kg_cols_to_drop = [c for c in composite.columns
                   if any(x in c.lower() for x in
                          ['kg_preds', 'kg_step', 'kg_domain',
                           'kg_clues', 'kg_votes', 'cleaned_combined_aspects_kg'])]
if kg_cols_to_drop:
    composite = composite.drop(columns=kg_cols_to_drop)
    print(f"Dropped old KG columns: {kg_cols_to_drop}")

# Re-split after dropping
mabsa_ar = composite[composite['dataset'].str.lower().str.contains('mabsa|m-absa', na=False)].copy().reset_index(drop=True)
se16_ar  = composite[composite['dataset'].str.lower().str.contains('semeval|se16|2016', na=False)].copy().reset_index(drop=True)
haad_ar  = composite[composite['dataset'].str.lower().str.contains('haad', na=False)].copy().reset_index(drop=True)

Dropped old KG columns: ['kg_preds_s2', 'kg_domain_s2', 'kg_step_s2', 'kg_clues_s2', 'kg_votes_s2', 'kg_preds', 'kg_domain', 'kg_step', 'kg_clues', 'cleaned_combined_aspects_kg_ar']


## STRATEGY 1 — Cross-lingual English KG Inference
**For**: M-ABSA Arabic, SemEval2016 Arabic
**Gold category format**: `ENTITY#ASPECT` pairs in English
**Method**: Arabic adjectives → multilingual embeddings → nearest English KG node → aspect lookup


### Block 6 — Load and Index English KG (Strategy 1)

In [ ]:
import pickle
from collections import defaultdict

# ── Load English KG JSON ───────────────────────────────────────────
with open(ENG_KG_PATH, 'r', encoding='utf-8') as f:
    raw_kg = json.load(f)

def _norm_eng(s: str) -> str:
    if not isinstance(s, str): return ''
    return re.sub(r'\s+', ' ', s.strip().lower())

# ── clue -> aspects (describes_aspect only) ────────────────────────
eng_clue_to_aspects: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in raw_kg.get('relationships', []):
    if r.get('predicate') != 'describes_aspect': continue
    clue = _norm_eng(r.get('subject', ''))
    asp  = _norm_eng(r.get('object',  ''))
    if clue and asp:
        entry = (asp, r['predicate'])
        if entry not in eng_clue_to_aspects[clue]:
            eng_clue_to_aspects[clue].append(entry)

# ── aspect -> entities ─────────────────────────────────────────────
eng_aspect_to_entities: Dict[str, Set[str]] = defaultdict(set)
for a in raw_kg.get('aspects', []):
    asp = _norm_eng(a.get('name', ''))
    for e in a.get('entity', []):
        eng_aspect_to_entities[asp].add(_norm_eng(e))

# ── entity -> domain ───────────────────────────────────────────────
eng_entity_to_domain: Dict[str, str] = {}
for e in raw_kg.get('entities', []):
    eng_entity_to_domain[_norm_eng(e['name'])] = e['domain'].strip().lower()

# ── aspect -> domains (derived) ────────────────────────────────────
eng_aspect_to_domains: Dict[str, Set[str]] = defaultdict(set)
for asp, ents in eng_aspect_to_entities.items():
    for ent in ents:
        dom = eng_entity_to_domain.get(ent)
        if dom:
            eng_aspect_to_domains[asp].add(dom)

print(f'✅ English KG loaded:')
print(f'   Clue entries  : {len(eng_clue_to_aspects)}')
print(f'   Aspect entries: {len(eng_aspect_to_entities)}')


# ================================================================
# Load or re-encode the KG adjective embeddings.
#
# The pkl may not exist or may use different key names depending on
# which version of the English pipeline notebook saved it.
# We handle all cases automatically:
#   1. pkl exists with known keys    -> load directly
#   2. pkl exists with unknown keys  -> print keys, re-encode from KG
#   3. pkl does not exist            -> encode from KG clues and save
#
# adj_labels     : List[str]       — one entry per KG clue
# adj_emb_matrix : np.ndarray (N, 384) — multilingual embeddings
# ================================================================

def _encode_clues_from_kg() -> None:
    """Encode all KG clue labels with the multilingual encoder and save."""
    global adj_labels, adj_emb_matrix, adj_node_ids
    import os
    adj_labels_to_encode = list(eng_clue_to_aspects.keys())
    print(f'  Encoding {len(adj_labels_to_encode)} clue labels '
          f'(~30-60 sec on GPU)...')
    adj_emb_matrix_enc = encoder.encode(
        adj_labels_to_encode,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
        batch_size=256,
    )
    os.makedirs(os.path.dirname(ENG_KG_ENCODED), exist_ok=True)
    with open(ENG_KG_ENCODED, 'wb') as fout:
        pickle.dump({
            'adj_labels':     adj_labels_to_encode,
            'adj_emb_matrix': adj_emb_matrix_enc,
            'adj_node_ids':   adj_labels_to_encode,
        }, fout)
    adj_labels     = adj_labels_to_encode
    adj_emb_matrix = adj_emb_matrix_enc
    adj_node_ids   = adj_labels_to_encode
    print(f'✅ Encoded and saved: {len(adj_labels)} nodes, '
          f'matrix shape {adj_emb_matrix.shape}')


adj_labels, adj_emb_matrix, adj_node_ids = [], np.array([]), []

try:
    with open(ENG_KG_ENCODED, 'rb') as f:
        encoded_kg = pickle.load(f)

    print(f'  pkl found. Keys in file: {list(encoded_kg.keys())}')

    # Try all known key name variants used across pipeline versions
    _LABEL_KEYS  = ['adj_labels', 'labels', 'node_labels',
                    'clue_labels', 'keys', 'words']
    _MATRIX_KEYS = ['adj_emb_matrix', 'embeddings', 'matrix',
                    'emb_matrix', 'vectors', 'embs']

    _found_labels = next(
        (encoded_kg[k] for k in _LABEL_KEYS  if k in encoded_kg), None)
    _found_matrix = next(
        (encoded_kg[k] for k in _MATRIX_KEYS if k in encoded_kg), None)

    if _found_labels is not None and _found_matrix is not None:
        adj_labels     = _found_labels
        adj_emb_matrix = _found_matrix
        adj_node_ids   = encoded_kg.get('adj_node_ids', adj_labels)
        print(f'✅ Encoded KG loaded: {len(adj_labels)} nodes, '
              f'matrix shape {adj_emb_matrix.shape}')
    else:
        print(f'⚠ pkl keys not recognised: {list(encoded_kg.keys())}')
        print('  Re-encoding from KG clues...')
        _encode_clues_from_kg()

except FileNotFoundError:
    print(f'⚠ pkl not found at {ENG_KG_ENCODED}')
    print('  Encoding from KG clues...')
    _encode_clues_from_kg()

# Final safety guard
assert len(adj_labels) > 0 and adj_emb_matrix.ndim == 2, (
    'adj_emb_matrix is still empty. '
    'Make sure encoder was loaded successfully in Block 2.'
)
print(f'✅ Ready: {len(adj_labels)} clue embeddings, '
      f'shape {adj_emb_matrix.shape}')

✅ English KG loaded:
   Clue entries  : 1114
   Aspect entries: 133
  pkl found. Keys in file: ['adj_labels', 'adj_emb_matrix', 'adj_node_ids']
✅ Encoded KG loaded: 1114 nodes, matrix shape (1114, 384)
✅ Ready: 1114 clue embeddings, shape (1114, 384)


### Block 6b — Encode English KG Adjectives (run ONCE if pkl not found)

In [ ]:
# ================================================================
# Block 6b — Re-encode English KG adjectives and overwrite the pkl
#
# Run this block ONLY if:
#   - Block 6 printed a "pkl keys not recognised" warning, OR
#   - You want to force a fresh re-encoding
#
# If Block 6 completed successfully with a green tick, skip this.
# ================================================================
import os, pickle

print('Force re-encoding English KG adjective nodes...')

adj_labels_to_encode = list(eng_clue_to_aspects.keys())
print(f'  Clues to encode: {len(adj_labels_to_encode)}')

adj_emb_matrix_enc = encoder.encode(
    adj_labels_to_encode,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=256,
)

os.makedirs(os.path.dirname(ENG_KG_ENCODED), exist_ok=True)

with open(ENG_KG_ENCODED, 'wb') as f:
    pickle.dump({
        'adj_labels':     adj_labels_to_encode,
        'adj_emb_matrix': adj_emb_matrix_enc,
        'adj_node_ids':   adj_labels_to_encode,
    }, f)

# Update the live variables so Block 7+ can use them immediately
adj_labels     = adj_labels_to_encode
adj_emb_matrix = adj_emb_matrix_enc
adj_node_ids   = adj_labels_to_encode

print(f'✅ Saved to {ENG_KG_ENCODED}')
print(f'   Nodes : {len(adj_labels)}')
print(f'   Shape : {adj_emb_matrix.shape}')
print(f'   Keys  : adj_labels, adj_emb_matrix, adj_node_ids')

Force re-encoding English KG adjective nodes...
  Clues to encode: 1114


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Saved to /content/drive/MyDrive/colab_files/encoded_kg/encoded_kg_minilm.pkl
   Nodes : 1114
   Shape : (1114, 384)
   Keys  : adj_labels, adj_emb_matrix, adj_node_ids


### Block 7 — Domain Configuration and Lookup Function (Strategy 1)

In [ ]:
# ── Domain aliases for English KG ────────────────────────────────
ENG_DOMAIN_ALIASES = {
    'restaurant': {'restaurant', 'food'},
    'food':       {'food', 'restaurant'},
    'hotel':      {'hotel'},
    'laptop':     {'laptop'},
    'phone':      {'phone'},
    'coursera':   {'coursera'},
    'sight':      {'sight'},
    'books':      {'books'},
}

S1_SIM_THRESHOLD    = 0.75   # Arabic adj → English adj cosine threshold
S1_MIN_ASPECT_SCORE = 0.65   # Min combined score to accept an aspect
S1_GENERIC_THRESHOLD = 0.85  # Higher bar for generic adjectives


def _filter_aspects_by_domain_eng(aspects, domain, asp_to_domains):
    allowed = {d.lower() for d in ENG_DOMAIN_ALIASES.get(domain.lower(), {domain.lower()})}
    filtered, unlinked = [], []
    for asp in aspects:
        asp_doms = asp_to_domains.get(asp, set())
        if not asp_doms:
            unlinked.append(asp)
        elif {d.lower() for d in asp_doms} & allowed:
            filtered.append(asp)
    return filtered if filtered else unlinked


def _lookup_adj_strategy1(adj_info: Dict, sentence: str, domain: str) -> Dict:
    """
    Cross-lingual: Arabic adj → multilingual embedding space →
    nearest English KG adj → linked aspects → domain filter → rank.
    """
    surface    = adj_info.get('surface', '')
    is_generic = adj_info.get('is_generic', False)
    lookup_key = adj_info.get('lemma_camel', '') or adj_info.get('surface', '')
    if not lookup_key:
        return {'surface': surface, 'final_aspects': [], 'final_scores': [],
                'reason': 'empty_lookup_key'}

    effective_thresh = S1_GENERIC_THRESHOLD if is_generic else S1_SIM_THRESHOLD

    ar_emb   = encoder.encode(lookup_key, normalize_embeddings=True, convert_to_numpy=True)
    sims     = np.dot(adj_emb_matrix, ar_emb)
    top5_idx = np.argsort(sims)[::-1][:5]
    top5     = [(adj_labels[i], float(sims[i])) for i in top5_idx]

    above = [(lbl, s) for lbl, s in top5 if s >= effective_thresh]
    if not above:
        return {'surface': surface,
                'top_en_match': top5[0][0] if top5 else None,
                'top_sim': round(top5[0][1], 3) if top5 else 0.0,
                'final_aspects': [], 'final_scores': [],
                'reason': f'below_threshold_{effective_thresh}'}

    # Collect aspects from matching English clue nodes
    asp_scores: Dict[str, float] = {}
    for en_lbl, sim in above:
        for asp, _ in eng_clue_to_aspects.get(en_lbl, []):
            if asp not in asp_scores or sim > asp_scores[asp]:
                asp_scores[asp] = sim

    if not asp_scores:
        return {'surface': surface, 'top_en_match': above[0][0],
                'top_sim': round(above[0][1], 3),
                'final_aspects': [], 'final_scores': [],
                'reason': 'no_aspects_in_kg'}

    # Domain filter
    filtered = _filter_aspects_by_domain_eng(
        list(asp_scores.keys()), domain, eng_aspect_to_domains
    )
    asp_scores = {a: s for a, s in asp_scores.items() if a in filtered}
    if not asp_scores:
        return {'surface': surface, 'top_en_match': above[0][0],
                'top_sim': round(above[0][1], 3),
                'final_aspects': [], 'final_scores': [],
                'reason': 'domain_filter_removed_all'}

    # Rank by sentence ↔ aspect cosine similarity
    asp_list  = list(asp_scores.keys())
    asp_embs  = encoder.encode(asp_list, normalize_embeddings=True, convert_to_numpy=True)
    sent_emb  = encoder.encode(sentence,  normalize_embeddings=True, convert_to_numpy=True)
    sent_sims = np.dot(asp_embs, sent_emb)

    combined_scores = []
    for i, asp in enumerate(asp_list):
        combined = 0.6 * asp_scores[asp] + 0.4 * float(sent_sims[i])
        if combined >= S1_MIN_ASPECT_SCORE:
            combined_scores.append((asp, round(combined, 3)))
    combined_scores.sort(key=lambda x: x[1], reverse=True)

    # Generic gate: only allow 'general'
    if is_generic:
        combined_scores = [(a, s) for a, s in combined_scores if a == 'general']

    return {
        'surface':       surface,
        'top_en_match':  above[0][0],
        'top_sim':       round(above[0][1], 3),
        'final_aspects': [a for a, _ in combined_scores],
        'final_scores':  combined_scores,
        'reason':        'ok' if combined_scores else 'post_filter_removed_all',
    }


print("✅ Strategy 1 domain config and lookup ready.")


✅ Strategy 1 domain config and lookup ready.


### Block 8 — Per-row Inference (Strategy 1)

In [ ]:
def infer_row_strategy1(row: pd.Series, max_aspects: int = 2) -> Dict:
    """
    Strategy 1: Arabic → cross-lingual English KG → English aspect terms.
    Gate: lonely adjective detection → embedding lookup → English KG.
    """
    sentence = str(row.get('sentence', '') or '').strip()
    if not sentence:
        return {'kg_preds': [], 'kg_domain': 'unknown',
                'kg_step': 'STOPPED_empty_sentence',
                'kg_clues': [], 'kg_lookups': {}}

    lonely_adjs = find_lonely_adjectives_ar(sentence)
    if not lonely_adjs:
        return {'kg_preds': [], 'kg_domain': 'unknown',
                'kg_step': 'STOPPED_no_lonely_adj',
                'kg_clues': [], 'kg_lookups': {}}

    inferred_domain = infer_domain_from_sentence(sentence, encoder)

    all_scores: Dict[str, float] = {}
    debug_lookups = {}
    for adj in lonely_adjs:
        result = _lookup_adj_strategy1(adj, sentence, inferred_domain)
        debug_lookups[adj['surface']] = result
        for asp, score in result.get('final_scores', []):
            if asp not in all_scores or score > all_scores[asp]:
                all_scores[asp] = score

    if not all_scores:
        return {'kg_preds': ['general'], 'kg_domain': inferred_domain,
                'kg_step': 'FALLBACK_general',
                'kg_clues': [a['surface'] for a in lonely_adjs],
                'kg_lookups': debug_lookups}

    ranked   = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)
    specific = [a for a, _ in ranked if a != 'general']
    final    = specific[:max_aspects] if specific else ['general']

    return {
        'kg_preds':   final,
        'kg_domain':  inferred_domain,
        'kg_step':    'OK',
        'kg_clues':   [a['surface'] for a in lonely_adjs],
        'kg_lookups': debug_lookups,
    }


print("✅ Strategy 1 per-row inference ready.")


✅ Strategy 1 per-row inference ready.


### Block 9 — Run Strategy 1 Inference

In [ ]:
from google.colab import files

def run_strategy1(df: pd.DataFrame, label: str, out_path: str) -> pd.DataFrame:
    print(f"\n📊 Running Strategy 1 (Cross-lingual English KG) on {label} ({len(df)} rows)...")
    results = df.progress_apply(infer_row_strategy1, axis=1)
    df = df.copy()
    df['kg_preds']  = results.apply(lambda d: d['kg_preds'])
    df['kg_domain'] = results.apply(lambda d: d['kg_domain'])
    df['kg_step']   = results.apply(lambda d: d['kg_step'])
    df['kg_clues']  = results.apply(lambda d: d['kg_clues'])
    print(f"\nStep distribution ({label}):")
    print(df['kg_step'].value_counts())
    if 'type' in df.columns:
        print(f"\nStep × Type ({label}):")
        print(pd.crosstab(df['type'], df['kg_step']))
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"✅ Saved: {out_path}")
    files.download(out_path)
    return df

mabsa_ar = run_strategy1(mabsa_ar, "M-ABSA Arabic",    "KG_Strategy1_MABSA_AR_predictions.csv")

if len(se16_ar) > 0:
    se16_ar = run_strategy1(se16_ar, "SemEval2016 Arabic", "KG_Strategy1_SemEval16_AR_predictions.csv")
else:
    print("⚠ No SemEval2016 Arabic rows — check split in Block 5.")



📊 Running Strategy 1 (Cross-lingual English KG) on M-ABSA Arabic (2182 rows)...


100%|██████████| 2182/2182 [20:33<00:00,  1.77it/s]



Step distribution (M-ABSA Arabic):
kg_step
STOPPED_no_lonely_adj    1159
FALLBACK_general          743
OK                        280
Name: count, dtype: int64

Step × Type (M-ABSA Arabic):
kg_step    FALLBACK_general   OK  STOPPED_no_lonely_adj
type                                                   
explicit                491  171                    694
implicit                200   86                    432
mixed                    52   23                     29
objective                 0    0                      4
✅ Saved: KG_Strategy1_MABSA_AR_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📊 Running Strategy 1 (Cross-lingual English KG) on SemEval2016 Arabic (1226 rows)...


100%|██████████| 1226/1226 [18:36<00:00,  1.10it/s]


Step distribution (SemEval2016 Arabic):
kg_step
FALLBACK_general         636
STOPPED_no_lonely_adj    475
OK                       115
Name: count, dtype: int64

Step × Type (SemEval2016 Arabic):
kg_step   FALLBACK_general   OK  STOPPED_no_lonely_adj
type                                                  
explicit               597  109                    424
implicit                39    6                     51
✅ Saved: KG_Strategy1_SemEval16_AR_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## STRATEGY 2 — Arabic Domain-Specific KG Inference
**For**: ALL datasets — M-ABSA Arabic, SemEval2016 Arabic, HAAD
**Gold category format**:
- M-ABSA / SemEval2016: `ENTITY#ASPECT` (English) — Strategy 2 predictions are Arabic, evaluated on aspect component
- HAAD: Arabic aspect strings only
**Method**: Arabic adjectives → direct Arabic KG lexical lookup → Arabic aspect terms


### Block 10 — Load and Index Arabic KG (Strategy 2)

In [ ]:
import json, re
from collections import defaultdict


def _extract_aspect(obj: str) -> str:
    """
    Extract the ASPECT part from an English KG object label.
    'food quality'           -> 'quality'
    'food general'           -> 'general'
    'laptop#general'         -> 'general'
    'laptop#design_features' -> 'design features'
    'GENERAL'                -> 'general'
    'overall#overall'        -> 'overall'
    """
    obj = obj.strip()
    if '#' in obj:
        return obj.split('#', 1)[1].strip().lower().replace('_', ' ')
    elif ' ' in obj:
        return obj.strip().split()[-1].strip().lower()
    else:
        return obj.strip().lower()


def load_arabic_kg(kg_path: str) -> dict:
    """
    Load and index the Arabic KG.

    Two edge types in the KG:
      - English-object edges (M-ABSA train splits):
            clue -> English aspect label  e.g. 'food quality' -> 'quality'
            These directly match the gold ENTITY#ASPECT format.
      - Arabic-object edges (SemEval2016 XML / HAAD / manual):
            clue -> Arabic surface noun   e.g. jayyid -> khidma
            Resolved at lookup time via ARABIC_NOUN_TO_ASPECT table.

    Both indexed separately, merged in lookup_clue_ar.
    """
    with open(kg_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    edges    = [e for e in raw.get('edges', [])
                if isinstance(e, dict) and 'predicate' in e]
    clues    = raw.get('clues', [])
    aspects  = raw.get('aspects', [])
    entities = raw.get('entities', [])

    # canonical -> all norm_ar alias forms
    canonical_to_norms: dict = defaultdict(set)
    for c in clues:
        canon = c.get('canonical', c.get('name', ''))
        forms = [c.get('name', ''), canon] + list(c.get('aliases', []))
        for f in forms:
            n = norm_ar(f)
            if n:
                canonical_to_norms[canon].add(n)

    # Index A: norm_ar(clue) -> [English aspect terms]  (directly usable)
    clue_to_eng_aspect: dict = defaultdict(list)

    # Index B: norm_ar(clue) -> [Arabic surface nouns]  (needs mapping)
    clue_to_ar_noun: dict = defaultdict(list)

    for e in edges:
        subj  = e.get('subject', '')
        obj   = str(e.get('object', ''))
        norms = canonical_to_norms.get(subj, {norm_ar(subj)})

        if re.search(r'[\u0600-\u06FF]', obj):
            # Arabic object -> store as noun, mapped later
            for n in norms:
                if n and obj not in clue_to_ar_noun[n]:
                    clue_to_ar_noun[n].append(obj)
        else:
            # English object -> extract aspect term directly
            asp = _extract_aspect(obj)
            if asp:
                for n in norms:
                    if n and asp not in clue_to_eng_aspect[n]:
                        clue_to_eng_aspect[n].append(asp)

    # aspect -> entities
    aspect_to_entities: dict = defaultdict(set)
    for a in aspects:
        if not isinstance(a, dict): continue
        asp_name = a.get('name', '')
        for ent in a.get('entity', []):
            aspect_to_entities[asp_name].add(ent)
        for alias in a.get('aliases', []):
            for ent in a.get('entity', []):
                aspect_to_entities[alias].add(ent)

    # entity -> domain
    entity_to_domain: dict = {}
    for e in entities:
        if not isinstance(e, dict): continue
        name = e.get('name', '')
        dom  = e.get('domain', '')
        for form in [name, name.lower(), name.upper()]:
            entity_to_domain[form] = dom

    # aspect -> domains (derived)
    aspect_to_domains: dict = defaultdict(set)
    for asp, ents in aspect_to_entities.items():
        for ent in ents:
            dom = (entity_to_domain.get(ent)
                   or entity_to_domain.get(ent.upper())
                   or entity_to_domain.get(ent.lower()))
            if dom:
                aspect_to_domains[asp].add(dom.lower())

    clue_set = set(clue_to_eng_aspect.keys()) | set(clue_to_ar_noun.keys())

    print(f'\u2705 Arabic KG loaded:')
    print(f'   English-aspect clue entries : {len(clue_to_eng_aspect)}')
    print(f'   Arabic-noun clue entries    : {len(clue_to_ar_noun)}')
    print(f'   Total clue set              : {len(clue_set)}')
    print(f'   Aspect entries              : {len(aspect_to_entities)}')
    print(f'   Valid edges                 : {len(edges)}')

    test_words = ['\u0646\u0638\u064a\u0641','\u0631\u0627\u064a\u0639',
                  '\u063a\u0627\u0644\u064a','\u0644\u0630\u064a\u0630',
                  '\u062c\u064a\u062f\u0647','\u0633\u064a\u0621',
                  '\u0645\u0631\u064a\u062d','\u0645\u0645\u062a\u0627\u0632']
    print('\n   Smoke test (clue -> eng_aspects | ar_nouns):')
    for w in test_words:
        n   = norm_ar(w)
        eng = clue_to_eng_aspect.get(n, [])
        ar  = clue_to_ar_noun.get(n, [])
        print(f"     '{w}' (norm='{n}') -> eng={eng[:3]} | ar={ar[:3]}")

    return {
        'raw':                raw,
        'clue_to_eng_aspect': dict(clue_to_eng_aspect),
        'clue_to_ar_noun':    dict(clue_to_ar_noun),
        'aspect_to_entities': dict(aspect_to_entities),
        'entity_to_domain':   entity_to_domain,
        'aspect_to_domains':  dict(aspect_to_domains),
        'clue_set':           clue_set,
    }


ar_kg_index = load_arabic_kg(AR_KG_PATH)


✅ Arabic KG loaded:
   English-aspect clue entries : 3431
   Arabic-noun clue entries    : 120
   Total clue set              : 3464
   Aspect entries              : 3341
   Valid edges                 : 5026

   Smoke test (clue -> eng_aspects | ar_nouns):
     'نظيف' (norm='نظيف') -> eng=['cleanliness'] | ar=[]
     'رايع' (norm='رايع') -> eng=['general', 'overall', 'aesthetics general'] | ar=[]
     'غالي' (norm='غالي') -> eng=['prices'] | ar=['اموال']
     'لذيذ' (norm='لذيذ') -> eng=['food_drinks'] | ar=[]
     'جيده' (norm='جيده') -> eng=['comfort', 'cleanliness', 'general'] | ar=['خدمة', 'اكل', 'اطلالة']
     'سيء' (norm='سيء') -> eng=['general', 'cleanliness', 'quality'] | ar=['غرفة', 'اثاث', 'شاليهة']
     'مريح' (norm='مريح') -> eng=['comfort'] | ar=[]
     'ممتاز' (norm='ممتاز') -> eng=['quality', 'overall'] | ar=[]


### Block 11 — Arabic KG Lookup and Per-row Inference (Strategy 2)

In [ ]:
# ================================================================
# ARABIC NOUN -> ENGLISH ASPECT mapping
# Resolves Arabic-object edges to English aspect terms
# matching the gold ENTITY#ASPECT evaluation format.
# ================================================================
ARABIC_NOUN_TO_ASPECT = {
    # Hotel / Restaurant
    'غرفة':   'general',
    'خدمة':   'general',
    'فندق':   'general',
    'مطعم':   'general',
    'اكل':         'quality',
    'طعام':   'quality',
    'طعم':         'quality',
    'وجبة':   'quality',
    'عشاء':   'quality',
    'فطور':   'quality',
    'افطار': 'quality',
    'سعر':         'prices',
    'اثاث':   'design features',
    'موقع':   'location',
    'استقبال': 'general',
    'موظف':   'general',
    'موظفين الاستقبال': 'general',
    'فريق العمل': 'general',
    'فريق':   'general',
    'طاقم':   'general',
    'تعامل': 'general',
    'معاملة': 'general',
    'عامل':   'general',
    'اقامة': 'general',
    'شاطي':   'general',
    'مسبح':   'general',
    'حمام سباحة': 'general',
    'حمامات السباحة': 'general',
    'مياه حمامات السباحة': 'general',
    'واي فاي': 'general',
    'تكييف': 'general',
    'كهرباء': 'general',
    'سرير':   'comfort',
    'دش':               'general',
    'نظافة': 'cleanliness',
    'مكان':   'location',
    'مدخل الفندق': 'general',
    'خدمة الغرف': 'general',
    'منتجع': 'general',
    'شاليهة': 'general',
    'فنادق': 'general',
    'مطاعم': 'general',
    'اموال': 'prices',
    'بالغرفة': 'general',
    'اطلالة': 'general',
    'فكرة':   'general',
    'الفكرة': 'general',
    # HAAD / Book domain
    'اسلوب': 'style options',
    'تقييم': 'general',
    'مؤلف':   'general',
    'مشاعر': 'general',
    'حبكه':   'general',
    'سلبيات': 'general',
    'مزايا': 'general',
    'قصة':         'general',
    'كتاب':   'general',
    'رواية': 'general',
    'لغة':         'style options',
    'كاتب':   'general',
    'سياق':   'general',
    'مضمون الكتاب': 'general',
    'صفحة':   'general',
}

# ================================================================
# FIX 2: Compound adjective-noun aspect mapping
# When an ADJ has an nmod NOUN child (e.g. سهلة الوصول),
# the noun component carries the aspect semantics.
# norm_ar(noun_component) -> aspect label
# ================================================================
COMPOUND_NOUN_TO_ASPECT = {
    # Accessibility / location
    'وصول':         'location',
    'موقع':         'location',
    'مكان':         'location',
    'بعد':          'location',
    'قرب':          'location',
    'مسافه':        'location',
    # Appearance / design
    'طله':          'design features',
    'مظهر':         'design features',
    'شكل':          'design features',
    'تصميم':        'design features',
    'ديكور':        'design features',
    'لون':          'design features',
    # Service / staff
    'استجابه':      'service',
    'تعامل':        'service',
    'خدمه':         'service',
    'معامله':       'service',
    'تواصل':        'service',
    'استقبال':      'service',
    # Cleanliness
    'نظافه':        'cleanliness',
    'تنظيف':        'cleanliness',
    'نظاف':         'cleanliness',
    # Comfort
    'راحه':         'comfort',
    'هدوء':         'comfort',
    'اتساع':        'comfort',
    # Quality / taste
    'طعم':          'quality',
    'جوده':         'quality',
    'نوعيه':        'quality',
    'مذاق':         'quality',
    # Price
    'سعر':          'prices',
    'تكلفه':        'prices',
    'اسعار':        'prices',
    'ثمن':          'prices',
    # Performance (laptop/phone)
    'اداء':         'operation performance',
    'سرعه':         'operation performance',
    # Content / material (coursera/books)
    'محتوي':        'quality',
    'ماده':         'quality',
    'محتوا':        'quality',
    'اسلوب':        'style options',
    'لغه':          'style options',
    # General
    'تجربه':        'general',
    'زياره':        'general',
    'اقامه':        'general',
}

# ================================================================
# EXTRA CLUES: adjectives detected but absent from KG clue set
# Issue 4 fix: tanwin forms + genuinely missing sentiment adjectives
# ================================================================
AR_EXTRA_CLUES = {
    # Generic positive -> general
    '\u062c\u064a\u062f\u0627':  'general',
    '\u062c\u064a\u062f\u0647':  'general',
    '\u0645\u0645\u062a\u0627\u0632\u0647': 'general',
    '\u0645\u0645\u062a\u0627\u0632\u0627': 'general',
    '\u0631\u0627\u064a\u0639\u0647': 'general',
    '\u0631\u0627\u064a\u0639\u0627': 'general',
    '\u0645\u062b\u0627\u0644\u064a': 'general',
    '\u0645\u062b\u0627\u0644\u064a\u0647': 'general',
    '\u0647\u0627\u064a\u0644':   'general',
    '\u062d\u0644\u0648':         'general',
    '\u062d\u0644\u0648\u0647':   'general',
    '\u0645\u0645\u064a\u0632':   'general',
    '\u0645\u0645\u064a\u0632\u0647': 'general',
    '\u0645\u0641\u0636\u0644':   'general',
    '\u0645\u0641\u0636\u0644\u0647': 'general',
    '\u062e\u0627\u0635':         'general',
    '\u062e\u0627\u0635\u0647':   'general',
    '\u0645\u062e\u062a\u0644\u0641': 'general',
    '\u0645\u062e\u062a\u0644\u0641\u0647': 'general',
    '\u0648\u0627\u0636\u062d':   'general',
    '\u0648\u0627\u0636\u062d\u0647': 'general',
    '\u0645\u0641\u064a\u062f':   'general',
    '\u0645\u0641\u064a\u062f\u0647': 'general',
    '\u0645\u0641\u064a\u062f\u0627': 'general',
    '\u062d\u0633\u0646':         'general',
    '\u062d\u0633\u0646\u0627':   'general',
    '\u0645\u0648\u062c\u0648\u062f': 'general',
    '\u0645\u0648\u062c\u0648\u062f\u0647': 'general',
    '\u0645\u0641\u0642\u0648\u062f': 'general',
    '\u0645\u0641\u0642\u0648\u062f\u0647': 'general',
    '\u0645\u062e\u064a\u0628':   'general',
    '\u0645\u062e\u064a\u0628\u0647': 'general',
    # Generic negative -> general
    '\u0633\u064a\u0626\u0647':   'general',
    '\u0633\u064a\u064a\u0647':   'general',
    '\u0633\u064a\u0626\u0627':   'general',
    '\u0631\u062f\u064a\u0626':   'general',
    '\u0631\u062f\u064a\u064a\u0647': 'general',
    '\u0631\u062f\u064a\u0626\u0647': 'general',
    '\u0648\u062d\u0634':         'general',
    '\u0636\u0639\u064a\u0641':   'general',
    '\u0636\u0639\u064a\u0641\u0647': 'general',
    # Texture / physical quality -> quality
    '\u0635\u0644\u0628':  'quality',
    '\u0635\u0644\u0628\u0647': 'quality',
    '\u0646\u0627\u0639\u0645': 'quality',
    '\u0646\u0627\u0639\u0645\u0647': 'quality',
    '\u0633\u0645\u064a\u0643': 'quality',
    '\u0633\u0645\u064a\u0643\u0647': 'quality',
    '\u0637\u0631\u064a': 'quality',
    '\u0637\u0631\u064a\u0647': 'quality',
    '\u0647\u0634': 'quality',
    '\u0647\u0634\u0647': 'quality',
    '\u0645\u0642\u0631\u0645\u0634': 'quality',
    # Size / portion -> style options
    '\u0643\u0628\u064a\u0631': 'style options',
    '\u0643\u0628\u064a\u0631\u0647': 'style options',
    '\u0635\u063a\u064a\u0631': 'style options',
    '\u0635\u063a\u064a\u0631\u0647': 'style options',
    '\u0645\u0645\u062a\u0644\u064a': 'style options',
    '\u0645\u0645\u062a\u0644\u064a\u0627': 'style options',
    # Price -> prices
    '\u063a\u0627\u0644\u064a\u0647': 'prices',
    '\u0631\u062e\u064a\u0635': 'prices',
    '\u0631\u062e\u064a\u0635\u0647': 'prices',
    '\u0645\u0639\u0642\u0648\u0644': 'prices',
    '\u0645\u0639\u0642\u0648\u0644\u0647': 'prices',
    # Comfort -> comfort
    '\u0645\u0631\u064a\u062d\u0647': 'comfort',
    '\u0645\u0631\u064a\u062d\u0627': 'comfort',
    '\u0647\u0627\u062f\u064a': 'comfort',
    '\u0647\u0627\u062f\u064a\u0647': 'comfort',
    '\u0647\u0627\u062f\u0626': 'comfort',
    '\u0647\u0627\u062f\u0626\u0647': 'comfort',
    '\u0647\u0627\u062f\u0626\u0627': 'comfort',
    # Cleanliness -> cleanliness
    '\u0646\u0638\u064a\u0641\u0647': 'cleanliness',
    '\u0646\u0638\u064a\u0641\u0627': 'cleanliness',
    '\u0645\u062a\u0633\u062e': 'cleanliness',
    '\u0645\u062a\u0633\u062e\u0647': 'cleanliness',
    '\u0648\u0633\u062e': 'cleanliness',
    '\u0648\u0633\u062e\u0647': 'cleanliness',
    # Speed / performance -> operation performance
    '\u0633\u0631\u064a\u0639\u0647': 'operation performance',
    '\u0633\u0631\u064a\u0639\u0627': 'operation performance',
    '\u0628\u0637\u064a\u0621': 'operation performance',
    '\u0628\u0637\u064a\u0626\u0647': 'operation performance',
    # Design / style -> design features
    '\u062c\u0645\u064a\u0644\u0647': 'design features',
    '\u0627\u0646\u064a\u0642': 'design features',
    '\u0627\u0646\u064a\u0642\u0647': 'design features',
    '\u0631\u0627\u0642\u064a\u0647': 'design features',
    '\u0639\u0635\u0631\u064a': 'design features',
    '\u0639\u0635\u0631\u064a\u0647': 'design features',
    '\u0642\u062f\u064a\u0645\u0647': 'design features',
    '\u062c\u062f\u064a\u062f\u0647': 'design features',
    # Tanwin adverbial forms -> general
    '\u0628\u0639\u064a\u062f\u0627': 'general',
    '\u0628\u0639\u064a\u062f\u0647': 'general',
    '\u0645\u062e\u0637\u064a\u0627': 'general',
}

print(f'\u2705 AR_EXTRA_CLUES: {len(AR_EXTRA_CLUES)} entries')
print(f'✅ ARABIC_NOUN_TO_ASPECT: {len(ARABIC_NOUN_TO_ASPECT)} entries')
print(f'✅ COMPOUND_NOUN_TO_ASPECT: {len(COMPOUND_NOUN_TO_ASPECT)} entries')


print(f'\u2705 AR_EXTRA_CLUES: {len(AR_EXTRA_CLUES)} entries')

def _norm_asp_eval(s: str) -> str:
    """Normalise an aspect label for evaluation comparison."""
    _GENERAL_SYN = {'misc', 'miscellaneous', 'other', 'general', 'عام'}
    n = re.sub(r'[_\-]', ' ', str(s).strip().lower())
    return 'general' if n in _GENERAL_SYN else n

# ================================================================
# VERB / PHRASE CLUE FALLBACK  (Issue 2 fix)
# Covers implicit sentences with no detectable lonely adjective.
# ================================================================

AR_VERB_TIER1 = {
    # Recommendation positive
    '\u0645\u0648\u0635\u0649 \u0628\u0647':  'general',
    '\u0623\u0648\u0635\u064a\u0643':          'general',
    '\u0623\u0648\u0635\u064a':                'general',
    '\u0623\u0648\u0635\u0649':                'general',
    '\u0646\u0646\u0635\u062d':                'general',
    '\u0623\u0646\u0635\u062d':                'general',
    '\u0646\u0648\u0635\u064a':                'general',
    '\u062a\u0648\u0635\u064a':                'general',
    '\u064a\u064f\u0646\u0635\u062d':          'general',
    # Recommendation negative
    '\u063a\u064a\u0631 \u0645\u0648\u0635\u0649': 'general',
    '\u0644\u0627 \u0623\u0648\u0635\u064a':   'general',
    '\u0644\u0627 \u0623\u0646\u0635\u062d':   'general',
    '\u0644\u0627 \u064a\u0646\u0635\u062d':   'general',
    # Love / like strong forms
    '\u0623\u062d\u0628\u0647\u0627': 'general',
    '\u0623\u062d\u0628\u0647':       'general',
    '\u0623\u062d\u0628\u0647\u0645': 'general',
    '\u0623\u062d\u0628\u0628\u062a\u0647': 'general',
    '\u0623\u062d\u0628\u0628\u062a\u0647\u0627': 'general',
    '\u0623\u0639\u062c\u0628\u0646\u064a': 'general',
    '\u0623\u0639\u062c\u0628\u062a\u0646\u064a': 'general',
    '\u064a\u0639\u062c\u0628\u0646\u064a': 'general',
    '\u0645\u0639\u062c\u0628':       'general',
    '\u0645\u0639\u062c\u0628\u0629': 'general',
    # Hate / dislike
    '\u0623\u0643\u0631\u0647\u0647': 'general',
    '\u0623\u0643\u0631\u0647\u0647\u0627': 'general',
    '\u0644\u0645 \u064a\u0639\u062c\u0628\u0646\u064a': 'general',
    '\u0644\u0627 \u064a\u0639\u062c\u0628\u0646\u064a': 'general',
    # Disappointment
    '\u0645\u062e\u064a\u0628 \u0644\u0644\u0622\u0645\u0627\u0644': 'general',
    '\u062e\u064a\u0628\u0629 \u0627\u0644\u0623\u0645\u0644': 'general',
    '\u062e\u064a\u0628\u062a\u0646\u064a': 'general',
    '\u0623\u062d\u0628\u0637\u062a\u0646\u064a': 'general',
    '\u062e\u0630\u0644\u062a\u0646\u064a': 'general',
}

AR_VERB_TIER2 = {
    # Love / like weak
    '\u0623\u062d\u0628':  'general',
    '\u0623\u062d\u0628\u0628': 'general',
    '\u0646\u062d\u0628':  'general',
    '\u062a\u062d\u0628':  'general',
    '\u064a\u0639\u062c\u0628': 'general',
    '\u0646\u0639\u062c\u0628': 'general',
    # Dislike
    '\u0646\u0643\u0631\u0647': 'general',
    '\u0623\u0643\u0631\u0647': 'general',
    # Repurchase / return
    '\u0633\u0623\u0639\u0648\u062f':  'general',
    '\u0633\u0623\u0631\u062c\u0639':  'general',
    '\u0633\u0623\u0634\u062a\u0631\u064a': 'general',
    '\u0633\u0623\u0637\u0644\u0628':  'general',
    '\u0633\u0623\u0632\u0648\u0631\u0647': 'general',
    '\u0633\u0646\u0639\u0648\u062f':  'general',
    '\u0645\u0631\u0629 \u0623\u062e\u0631\u0649': 'general',
    '\u0645\u062c\u062f\u062f\u064b\u0627': 'general',
    '\u0645\u062c\u062f\u062f\u0627':  'general',
    '\u062b\u0627\u0646\u064a\u0629':  'general',
    # Worthwhile
    '\u064a\u0633\u062a\u062d\u0642': 'general',
    '\u062a\u0633\u062a\u062d\u0642': 'general',
    '\u0627\u0633\u062a\u062d\u0642': 'general',
    '\u064a\u0633\u062a\u0627\u0647\u0644': 'general',
    '\u062a\u0633\u062a\u0627\u0647\u0644': 'general',
}

AR_AMPLIFIERS = {
    '\u0628\u0627\u0644\u062a\u0623\u0643\u064a\u062f',
    '\u062d\u0642\u064b\u0627', '\u062d\u0642\u0627',
    '\u062c\u062f\u064b\u0627', '\u062c\u062f\u0627',
    '\u0644\u0644\u063a\u0627\u064a\u0629',
    '\u0643\u062b\u064a\u0631\u064b\u0627', '\u0643\u062b\u064a\u0631\u0627',
    '\u0628\u0634\u062f\u0629',
    '\u062a\u0645\u0627\u0645\u064b\u0627', '\u062a\u0645\u0627\u0645\u0627',
    '\u0641\u0639\u0644\u064b\u0627', '\u0641\u0639\u0644\u0627',
    '\u0628\u062d\u0642',
    '\u062d\u062a\u0645\u064b\u0627', '\u062d\u062a\u0645\u0627',
}

# Negation constructs — implicit negative sentiment
# 'لا يوجد X' / 'لم أواجه مشاكل' → general
# 'لا X' where X is a domain noun → maps to that domain's general/quality
AR_NEGATION_TIER1 = {
    'لا يوجد':     'general',    # there is no X
    'لا توجد':     'general',
    'لم يكن هناك': 'general',
    'لم أجد':      'general',
    'لم أواجه':    'general',
    'لم يعجبني':   'general',
    'لم تعجبني':   'general',
    'ليس جيد':     'quality',
    'ليست جيدة':   'quality',
    'ليس ممتاز':   'quality',
    'غير مريح':    'comfort',
    'غير نظيف':    'cleanliness',
    'غير مستساغ':  'quality',
    'لا أنصح':     'general',
    'لا ينصح':     'general',
    'لا تشتر':     'general',
    'لا تذهب':     'general',
    'تجنب':        'general',
    'تجنبه':       'general',
    'تجنبها':      'general',
}

AR_VERB_TIER1.update(AR_NEGATION_TIER1)
AR_COMPARATIVE_TIER2 = {
    'أفضل':   'quality',    # better → quality
    'أحسن':   'quality',
    'أسوأ':   'quality',
    'أردأ':   'quality',
    'أرخص':   'prices',     # cheaper → prices
    'أغلى':   'prices',     # more expensive → prices
    'أكثر':   'general',
    'أقل':    'general',
    'أكبر':   'general',
    'أصغر':   'general',
}
AR_VERB_TIER2.update(AR_COMPARATIVE_TIER2)

def _scan_verb_clues_ar(sentence: str):
    """
    Scan for Arabic verb/phrase sentiment clues.
    Returns: (aspect: str, tier: int)
      tier=0 -> no clue
      tier=1 -> strong signal (VERB_TIER1_general)
      tier=2 -> weak signal   (VERB_TIER2_general)
    """
    if not sentence:
        return 'general', 0
    has_amplifier = any(amp in sentence for amp in AR_AMPLIFIERS)
    for phrase in sorted(AR_VERB_TIER1, key=len, reverse=True):
        if phrase in sentence:
            return AR_VERB_TIER1[phrase], 1
    for phrase in sorted(AR_VERB_TIER2, key=len, reverse=True):
        if phrase in sentence:
            return AR_VERB_TIER2[phrase], (1 if has_amplifier else 2)
    return 'general', 0


# ================================================================
# DOMAIN ALIASES
# ================================================================
AR_DOMAIN_ALIASES = {
    'food':       {'food', 'restaurant'},
    'restaurant': {'restaurant', 'food'},
    'hotel':      {'hotel'},
    'hotels':     {'hotel'},
    'laptop':     {'laptop'},
    'phone':      {'phone'},
    'coursera':   {'coursera'},
    'sight':      {'sight'},
    'books':      {'books'},
}


def _match_ar_clue(adj_info: dict, kg_index: dict,
                   strategy: str = 'exact+prefix'):
    """
    Match Arabic adjective against KG clue set.
    FIX 1: adds masculine base reduction candidates after norm_ar forms.
    Order: exact (norm) -> exact (masc base) -> tanwin/alef strip ->
           prefix -> AR_EXTRA_CLUES -> substring
    """
    raw_forms = [
        adj_info.get('lemma_camel', ''),
        adj_info.get('surface', ''),
        adj_info.get('lemma_stanza', ''),
        adj_info.get('norm_ar', ''),
    ]
    norm_forms = []
    seen = set()
    for f in raw_forms:
        n = norm_ar(f)
        if n and n not in seen:
            seen.add(n); norm_forms.append(n)
        # Strip trailing taa marbuta -> haa variant
        n2 = re.sub(r'\u0647$', '', n).strip()
        if n2 and n2 != n and n2 not in seen:
            seen.add(n2); norm_forms.append(n2)
        # Tanwin fix: strip trailing adverbial alef
        n3 = re.sub(r'\u0627$', '', n).strip()
        if n3 and n3 != n and len(n3) > 3 and n3 not in seen:
            seen.add(n3); norm_forms.append(n3)

    # FIX 1: add masculine base reductions for all norm_forms
    surface = adj_info.get('surface', '')
    masculine_bases = []
    for n in list(norm_forms):
        base = reduce_to_masculine_base(n, surface=surface)
        if base != n and base not in seen:
            seen.add(base)
            masculine_bases.append(base)
    # Also try masc_base / masc_base_cam if precomputed in adj_info
    for key in ('masc_base', 'masc_base_cam'):
        b = adj_info.get(key, '')
        if b and b not in seen:
            seen.add(b)
            masculine_bases.append(b)

    clue_set = kg_index['clue_set']

    # Step 1: exact on original norm_forms
    if 'exact' in strategy:
        for cand in norm_forms:
            if cand in clue_set:
                return [cand], 'exact'

    # Step 2: exact on masculine bases (FIX 1)
    for cand in masculine_bases:
        if cand in clue_set:
            return [cand], 'exact_masculine_base'

    # Step 3: prefix match
    if 'prefix' in strategy:
        for cand in norm_forms + masculine_bases:
            if len(cand) < 3: continue
            matched = [k for k in clue_set if k.startswith(cand[:3])]
            if matched: return matched, 'prefix'

    # Step 4: AR_EXTRA_CLUES fallback
    for cand in norm_forms + masculine_bases:
        if cand in AR_EXTRA_CLUES:
            return [f'__EXTRA__:{AR_EXTRA_CLUES[cand]}'], 'extra_clue'

    # Step 5: substring match
    if 'substring' in strategy:
        for cand in norm_forms + masculine_bases:
            if not cand: continue
            matched = [k for k in clue_set if k in cand or cand in k]
            if matched: return matched, 'substring'

    return [], 'none'


# def _filter_aspects_by_domain_ar(aspects_list: list, domain: str,
#                                   kg_index: dict) -> list:
#     """Keep aspects in the inferred domain. Falls back to all if none survive."""
#     if not aspects_list or not domain:
#         return aspects_list
#     allowed = {d.lower() for d in AR_DOMAIN_ALIASES.get(domain.lower(), {domain})}
#     filtered, unlinked = [], []
#     for asp, pred in aspects_list:
#         asp_doms = {d.lower() for d in kg_index['aspect_to_domains'].get(asp, set())}
#         if not asp_doms:
#             unlinked.append((asp, pred))
#         elif asp_doms & allowed:
#             filtered.append((asp, pred))

#     return filtered if filtered else unlinked

def _filter_aspects_by_domain_ar(aspects_list: list, domain: str,
                                  kg_index: dict) -> list:
    """
    Keep only aspects that belong to the inferred domain.

    STRICT: if nothing survives domain filter, return [('general', ...)]
    as fallback instead of the unfiltered list.
    This prevents Phone-domain aspects (overall, running speed, loyalty)
    from leaking into hotel/food/restaurant queries.
    """
    if not aspects_list or not domain:
        return [('general', 'describes_aspect')]
    allowed = {d.lower() for d in AR_DOMAIN_ALIASES.get(domain.lower(), {domain})}
    filtered = []
    for asp, pred in aspects_list:
        asp_doms = {d.lower() for d in kg_index['aspect_to_domains'].get(asp, set())}
        if not asp_doms:
            # Aspect has no domain tag — only keep if it is a generic term
            if _norm_asp_eval(asp) in ('general', 'quality', 'prices',
                                        'comfort', 'cleanliness', 'location',
                                        'style options', 'design features'):
                filtered.append((asp, pred))
        elif asp_doms & allowed:
            filtered.append((asp, pred))
    # Strict fallback: never return unfiltered cross-domain aspects
    return filtered if filtered else [('general', 'describes_aspect')]

# def lookup_clue_ar(adj_info: dict, domain: str, kg_index: dict,
#                    match_strategy: str = 'exact+prefix') -> dict:
#     """
#     Arabic KG clue lookup. Merges three paths:
#       Path A: English-object edges  -> clue_to_eng_aspect (directly usable)
#       Path B: Arabic-object edges   -> clue_to_ar_noun via ARABIC_NOUN_TO_ASPECT
#       Path C: AR_EXTRA_CLUES hit    -> aspect encoded in match key
#     """
#     surface = adj_info.get('surface', '')
#     matched, match_type = _match_ar_clue(adj_info, kg_index, match_strategy)

#     if not matched:
#         return {'surface': surface, 'matched_clues': [], 'match_type': 'none',
#                 'raw_aspects': [], 'domain_aspects': [],
#                 'top_aspect': None, 'reason': 'no_clue_match'}

#     # Path C: extra clue hit
#     if match_type == 'extra_clue':
#         aspect = matched[0].split(':', 1)[1]
#         return {
#             'surface': surface, 'matched_clues': matched,
#             'match_type': 'extra_clue',
#             'raw_aspects': [(aspect, 'describes_aspect')],
#             'domain_aspects': [(aspect, 'describes_aspect')],
#             'top_aspect': aspect, 'reason': 'ok',
#         }

#     # Paths A + B: collect from both indexes
#     raw_aspects = []
#     for kg_clue in matched:
#         # Path A
#         for asp in kg_index['clue_to_eng_aspect'].get(kg_clue, []):
#             entry = (asp, 'describes_aspect')
#             if entry not in raw_aspects:
#                 raw_aspects.append(entry)
#         # Path B
#         for ar_noun in kg_index['clue_to_ar_noun'].get(kg_clue, []):
#             mapped = ARABIC_NOUN_TO_ASPECT.get(ar_noun)
#             if mapped:
#                 entry = (mapped, 'describes_aspect')
#                 if entry not in raw_aspects:
#                     raw_aspects.append(entry)

#     if not raw_aspects:
#         return {'surface': surface, 'matched_clues': matched,
#                 'match_type': match_type, 'raw_aspects': [],
#                 'domain_aspects': [], 'top_aspect': None,
#                 'reason': 'no_aspects_for_clue'}

#     domain_aspects = _filter_aspects_by_domain_ar(raw_aspects, domain, kg_index)
#     if not domain_aspects:
#         return {'surface': surface, 'matched_clues': matched,
#                 'match_type': match_type, 'raw_aspects': raw_aspects,
#                 'domain_aspects': [], 'top_aspect': None,
#                 'reason': 'domain_filter_removed_all'}

#     domain_entities = {
#         asp: kg_index['aspect_to_entities'].get(asp, set())
#         for asp, _ in domain_aspects
#     }
#     sorted_aspects = sorted(
#         domain_aspects,
#         key=lambda x: len(domain_entities.get(x[0], set())),
#         reverse=True,
#     )
#     return {
#         'surface': surface, 'matched_clues': matched,
#         'match_type': match_type, 'raw_aspects': raw_aspects,
#         'domain_aspects': sorted_aspects,
#         'top_aspect': sorted_aspects[0][0] if sorted_aspects else None,
#         'reason': 'ok',
#     }
def lookup_clue_ar(adj_info: dict, domain: str, kg_index: dict,
                   match_strategy: str = 'exact') -> dict:
    """
    Arabic KG clue lookup. Merges four paths:
      Path A: English-object edges  -> clue_to_eng_aspect (directly usable)
      Path B: Arabic-object edges   -> clue_to_ar_noun via ARABIC_NOUN_TO_ASPECT
      Path C: AR_EXTRA_CLUES hit    -> aspect encoded in match key
      Path D: Compound noun lookup  -> COMPOUND_NOUN_TO_ASPECT (FIX 2)

    Domain filtering is strict: cross-domain aspects are dropped and
    replaced with 'general' rather than returned unfiltered.
    """
    surface       = adj_info.get('surface', '')
    compound_noun = adj_info.get('compound_noun')

    # ── Path D: compound noun lookup (FIX 2) ────────────────────
    if compound_noun:
        noun_norm  = norm_ar(compound_noun)
        noun_norms = [noun_norm]
        # Also try without definite article
        if noun_norm.startswith('ال') and len(noun_norm) > 2:
            noun_norms.append(noun_norm[2:])
        for nn in noun_norms:
            asp = COMPOUND_NOUN_TO_ASPECT.get(nn)
            if asp:
                domain_aspects = _filter_aspects_by_domain_ar(
                    [(asp, 'compound_noun')], domain, kg_index
                )
                top = domain_aspects[0][0] if domain_aspects else asp
                return {
                    'surface':        surface,
                    'compound_noun':  compound_noun,
                    'matched_clues':  [nn],
                    'match_type':     'compound_noun',
                    'raw_aspects':    [(asp, 'compound_noun')],
                    'domain_aspects': domain_aspects,
                    'top_aspect':     top,
                    'reason':         'ok',
                }

    matched, match_type = _match_ar_clue(adj_info, kg_index, match_strategy)

    if not matched:
        return {'surface': surface, 'matched_clues': [], 'match_type': 'none',
                'raw_aspects': [], 'domain_aspects': [],
                'top_aspect': None, 'reason': 'no_clue_match'}

    # Path C: extra clue hit — aspect is encoded in the match key
    if match_type == 'extra_clue':
        aspect = matched[0].split(':', 1)[1]
        return {
            'surface': surface, 'matched_clues': matched,
            'match_type': 'extra_clue',
            'raw_aspects': [(aspect, 'describes_aspect')],
            'domain_aspects': [(aspect, 'describes_aspect')],
            'top_aspect': aspect, 'reason': 'ok',
        }

    # Paths A + B: collect from both indexes
    raw_aspects = []
    for kg_clue in matched:
        # Path A: English-aspect index
        for asp in kg_index['clue_to_eng_aspect'].get(kg_clue, []):
            entry = (asp, 'describes_aspect')
            if entry not in raw_aspects:
                raw_aspects.append(entry)
        # Path B: Arabic-noun index via ARABIC_NOUN_TO_ASPECT
        for ar_noun in kg_index['clue_to_ar_noun'].get(kg_clue, []):
            mapped = ARABIC_NOUN_TO_ASPECT.get(ar_noun)
            if mapped:
                entry = (mapped, 'describes_aspect')
                if entry not in raw_aspects:
                    raw_aspects.append(entry)

    if not raw_aspects:
        return {'surface': surface, 'matched_clues': matched,
                'match_type': match_type, 'raw_aspects': [],
                'domain_aspects': [], 'top_aspect': None,
                'reason': 'no_aspects_for_clue'}

    # Strict domain filter — never lets cross-domain aspects through
    domain_aspects = _filter_aspects_by_domain_ar(raw_aspects, domain, kg_index)

    # Rank: aspects with more entity associations ranked higher
    domain_entities = {
        asp: kg_index['aspect_to_entities'].get(asp, set())
        for asp, _ in domain_aspects
    }
    sorted_aspects = sorted(
        domain_aspects,
        key=lambda x: len(domain_entities.get(x[0], set())),
        reverse=True,
    )

    return {
        'surface':        surface,
        'matched_clues':  matched,
        'match_type':     match_type,
        'raw_aspects':    raw_aspects,
        'domain_aspects': sorted_aspects,
        'top_aspect':     sorted_aspects[0][0] if sorted_aspects else None,
        'reason':         'ok',
    }

# def infer_row_strategy2(row, kg_index: dict, max_aspects: int = 2,
#                         match_strategy: str = 'exact+prefix') -> dict:
#     """
#     Strategy 2 per-row inference.
#     Issue 2 fix: verb/phrase fallback when no lonely adjective found.
#     Issue 4 fix: tanwin stripping + AR_EXTRA_CLUES in _match_ar_clue.
#     """
#     sentence = str(row.get('sentence', '') or '').strip()
#     if not sentence:
#         return {'kg_preds': [], 'kg_domain': 'unknown',
#                 'kg_step': 'STOPPED_empty_sentence',
#                 'kg_clues': [], 'kg_lookups': {}, 'kg_votes': {}}

#     lonely_adjs = find_lonely_adjectives_ar(sentence)

#     if not lonely_adjs:
#         verb_aspect, verb_tier = _scan_verb_clues_ar(sentence)
#         if verb_tier == 1:
#             return {'kg_preds': [verb_aspect], 'kg_domain': 'unknown',
#                     'kg_step': 'VERB_TIER1_general',
#                     'kg_clues': [], 'kg_lookups': {}, 'kg_votes': {}}
#         if verb_tier == 2:
#             return {'kg_preds': [verb_aspect], 'kg_domain': 'unknown',
#                     'kg_step': 'VERB_TIER2_general',
#                     'kg_clues': [], 'kg_lookups': {}, 'kg_votes': {}}
#         return {'kg_preds': [], 'kg_domain': 'unknown',
#                 'kg_step': 'STOPPED_no_lonely_adj',
#                 'kg_clues': [], 'kg_lookups': {}, 'kg_votes': {}}

#     inferred_domain = infer_domain_from_sentence(sentence, encoder)

#     aspect_votes: dict = defaultdict(int)
#     debug_lookups = {}
#     for adj in lonely_adjs:
#         result = lookup_clue_ar(adj, inferred_domain, kg_index, match_strategy)
#         debug_lookups[adj['surface']] = result
#         for asp, _ in result.get('domain_aspects', []):
#             aspect_votes[asp] += 1

#     if not aspect_votes:
#         verb_aspect, verb_tier = _scan_verb_clues_ar(sentence)
#         if verb_tier >= 1:
#             return {'kg_preds': [verb_aspect], 'kg_domain': inferred_domain,
#                     'kg_step': f'VERB_TIER{verb_tier}_general',
#                     'kg_clues': [a['surface'] for a in lonely_adjs],
#                     'kg_lookups': debug_lookups, 'kg_votes': {}}
#         return {'kg_preds': [], 'kg_domain': inferred_domain,
#                 'kg_step': 'STOPPED_no_kg_match',
#                 'kg_clues': [a['surface'] for a in lonely_adjs],
#                 'kg_lookups': debug_lookups, 'kg_votes': {}}

#     ranked   = sorted(aspect_votes.items(), key=lambda x: x[1], reverse=True)
#     specific = [a for a, _ in ranked if a.lower() not in ('general', '\u0639\u0627\u0645')]
#     final    = specific[:max_aspects] if specific else [ranked[0][0]]

#     return {
#         'kg_preds':   final,
#         'kg_domain':  inferred_domain,
#         'kg_step':    'OK',
#         'kg_clues':   [a['surface'] for a in lonely_adjs],
#         'kg_lookups': debug_lookups,
#         'kg_votes':   dict(ranked),
#     }

# ================================================================
# Fix: use ground truth domain from row instead of inferring it
# Replace this line in infer_row_strategy2:
#
#   inferred_domain = infer_domain_from_sentence(sentence, encoder)
#
# With:
# ================================================================

def infer_row_strategy2(row, kg_index: dict, max_aspects: int = 2,
                        match_strategy: str = 'exact') -> dict:

    sentence = str(row.get('sentence', '') or '').strip()
    if not sentence:
        return {'kg_preds': [], 'kg_domain': 'unknown',
                'kg_step': 'STOPPED_empty_sentence',
                'kg_clues': [], 'kg_lookups': {}, 'kg_votes': {}}

    # Use ground truth domain directly — eliminates cross-domain errors
    domain = str(row.get('domain', '') or '').strip().lower()
    if not domain or domain == 'nan':
        # Fallback to inference only when domain is genuinely missing
        domain = infer_domain_from_sentence(sentence, encoder)
        domain_source = 'inferred'
    else:
        domain_source = 'ground_truth'

    lonely_adjs = find_lonely_adjectives_ar(sentence)

    if not lonely_adjs:
        verb_aspect, verb_tier = _scan_verb_clues_ar(sentence)
        if verb_tier == 1:
            return {'kg_preds': [verb_aspect], 'kg_domain': domain,
                    'kg_step': 'VERB_TIER1_general',
                    'kg_clues': [], 'kg_lookups': {}, 'kg_votes': {}}
        if verb_tier == 2:
            return {'kg_preds': [verb_aspect], 'kg_domain': domain,
                    'kg_step': 'VERB_TIER2_general',
                    'kg_clues': [], 'kg_lookups': {}, 'kg_votes': {}}
        return {'kg_preds': [], 'kg_domain': domain,
                'kg_step': 'STOPPED_no_lonely_adj',
                'kg_clues': [], 'kg_lookups': {}, 'kg_votes': {}}

    aspect_votes: dict = defaultdict(int)
    debug_lookups = {}
    for adj in lonely_adjs:
        result = lookup_clue_ar(adj, domain, kg_index, match_strategy)
        debug_lookups[adj['surface']] = result
        for asp, _ in result.get('domain_aspects', []):
            aspect_votes[asp] += 1

    if not aspect_votes:
        verb_aspect, verb_tier = _scan_verb_clues_ar(sentence)
        if verb_tier >= 1:
            return {'kg_preds': [verb_aspect], 'kg_domain': domain,
                    'kg_step': f'VERB_TIER{verb_tier}_general',
                    'kg_clues': [a['surface'] for a in lonely_adjs],
                    'kg_lookups': debug_lookups, 'kg_votes': {}}
        return {'kg_preds': [], 'kg_domain': domain,
                'kg_step': 'STOPPED_no_kg_match',
                'kg_clues': [a['surface'] for a in lonely_adjs],
                'kg_lookups': debug_lookups, 'kg_votes': {}}

    ranked   = sorted(aspect_votes.items(), key=lambda x: x[1], reverse=True)
    specific = [a for a, _ in ranked
                if a.lower() not in ('general', 'عام')]
    final    = specific[:max_aspects] if specific else [ranked[0][0]]

    return {
        'kg_preds':        final,
        'kg_domain':       domain,
        'kg_domain_source': domain_source,
        'kg_step':         'OK',
        'kg_clues':        [a['surface'] for a in lonely_adjs],
        'kg_lookups':      debug_lookups,
        'kg_votes':        dict(ranked),
    }

print("✅ infer_row_strategy2 updated: ground truth domain used directly.")

print('\u2705 Block 11 ready: FIX1 feminine reduction, FIX2 compound detection, verb fallback, tanwin fix.')


✅ AR_EXTRA_CLUES: 91 entries
✅ ARABIC_NOUN_TO_ASPECT: 62 entries
✅ COMPOUND_NOUN_TO_ASPECT: 42 entries
✅ AR_EXTRA_CLUES: 91 entries
✅ infer_row_strategy2 updated: ground truth domain used directly.
✅ Block 11 ready: FIX1 feminine reduction, FIX2 compound detection, verb fallback, tanwin fix.


### Block 12 — Run Strategy 2 Inference (ALL datasets)

In [ ]:
from google.colab import files

# ================================================================
# TEST MODE — set True to run on a small subset for quick checks.
# CSVs are always saved regardless of mode for error analysis.
# Set TEST_MODE = False for the full experiment.
# ================================================================
TEST_MODE   = False #True will run experiments on a subset of "TEST_N_ROWS"
TEST_N_ROWS = 150   # rows per dataset in test mode

# def _sample(df: pd.DataFrame, n: int) -> pd.DataFrame:
#     """Sample n rows stratified by type if possible."""
#     if 'type' not in df.columns or len(df) <= n:
#         return df.head(n)
#     sampled = (df.groupby('type', group_keys=False)
#                  .apply(lambda g: g.head(max(1, int(n * len(g) / len(df))))))
#     return sampled.head(n).reset_index(drop=True)

def _sample(df: pd.DataFrame, n: int) -> pd.DataFrame:
    """
    Sample n implicit rows + an equal number of non-implicit rows.
    This gives a meaningful implicit evaluation while keeping
    the run short. Falls back to full df if not enough rows.
    """
    if 'type' not in df.columns:
        return df.head(n)

    impl     = df[df['type'] == 'implicit']
    non_impl = df[df['type'] != 'implicit']

    # Take up to n implicit rows
    impl_sample = impl.head(n) if len(impl) > n else impl

    # Take same number of non-implicit rows to preserve context
    n_non = min(len(non_impl), len(impl_sample))
    non_impl_sample = non_impl.head(n_non)

    result = pd.concat([impl_sample, non_impl_sample]).reset_index(drop=True)
    print(f"  Sample: {len(impl_sample)} implicit + {n_non} non-implicit "
          f"= {len(result)} rows total")
    return result
def run_strategy2(df: pd.DataFrame, label: str, out_path: str) -> pd.DataFrame:
    if TEST_MODE:
        df_run = _sample(df, TEST_N_ROWS)
        print(f'\n\U0001f4ca [TEST MODE: {len(df_run)}/{len(df)} rows] '
              f'Strategy 2 on {label}...')
    else:
        df_run = df
        print(f'\n\U0001f4ca Running Strategy 2 (Arabic KG) on {label} '
              f'({len(df)} rows)...')

    results = df_run.progress_apply(
        lambda row: infer_row_strategy2(row, ar_kg_index),
        axis=1
    )
    df_run = df_run.copy()

    df_run['kg_preds_s2']  = results.apply(lambda d: d['kg_preds'])
    df_run['kg_domain_s2'] = results.apply(lambda d: d['kg_domain'])
    df_run['kg_step_s2']   = results.apply(lambda d: d['kg_step'])
    df_run['kg_clues_s2']  = results.apply(lambda d: d['kg_clues'])
    df_run['kg_votes_s2']  = results.apply(lambda d: d.get('kg_votes', {}))

    print(f'\nStep distribution ({label}):')
    print(df_run['kg_step_s2'].value_counts())
    if 'type' in df_run.columns:
        print(f'\nStep x Type ({label}):')
        print(pd.crosstab(df_run['type'], df_run['kg_step_s2']))

    # ── Always save CSV for error analysis ───────────────────────
    # In TEST_MODE the filename gets a _test suffix so it does not
    # overwrite a previous full-run CSV saved at the same path.
    save_path = out_path.replace('.csv', '_test.csv') if TEST_MODE else out_path
    df_run.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f'\u2705 Saved: {save_path} ({len(df_run)} rows)')
    files.download(save_path)

    if TEST_MODE:
        print('  [TEST MODE] Set TEST_MODE=False and re-run for full experiment.')

    return df_run


mabsa_ar_out = run_strategy2(
    mabsa_ar, 'M-ABSA Arabic',
    'KG_Strategy2_MABSA_AR_predictions.csv'
)

if len(se16_ar) > 0:
    se16_ar_out = run_strategy2(
        se16_ar, 'SemEval2016 Arabic',
        'KG_Strategy2_SemEval16_AR_predictions.csv'
    )
else:
    print('\u26a0 No SemEval2016 Arabic rows — check split in Block 5.')
    se16_ar_out = se16_ar

if len(haad_ar) > 0:
    haad_ar_out = run_strategy2(
        haad_ar, 'HAAD',
        'KG_Strategy2_HAAD_predictions.csv'
    )
else:
    print('\u26a0 No HAAD rows — check split in Block 5.')
    haad_ar_out = haad_ar

print('\n\u2705 Block 12 complete.')
print('   Full dataframes: mabsa_ar, se16_ar, haad_ar (unchanged)')
print('   Run dataframes : mabsa_ar_out, se16_ar_out, haad_ar_out')
print('   Evaluation block will automatically use the run dataframes.')



📊 Running Strategy 2 (Arabic KG) on M-ABSA Arabic (2182 rows)...


100%|██████████| 2182/2182 [15:26<00:00,  2.35it/s]



Step distribution (M-ABSA Arabic):
kg_step_s2
STOPPED_no_lonely_adj    798
OK                       792
VERB_TIER2_general       248
VERB_TIER1_general       176
STOPPED_no_kg_match      168
Name: count, dtype: int64

Step x Type (M-ABSA Arabic):
kg_step_s2   OK  STOPPED_no_kg_match  STOPPED_no_lonely_adj  \
type                                                          
explicit    526                  102                    479   
implicit    204                   55                    295   
mixed        62                   11                     22   
objective     0                    0                      2   

kg_step_s2  VERB_TIER1_general  VERB_TIER2_general  
type                                                
explicit                    88                 161  
implicit                    84                  80  
mixed                        4                   5  
objective                    0                   2  
✅ Saved: KG_Strategy2_MABSA_AR_predictions.csv (2182 ro

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

100%|██████████| 299/299 [03:34<00:00,  1.39it/s]


Step distribution (HAAD):
kg_step_s2
STOPPED_no_lonely_adj    129
OK                        78
STOPPED_no_kg_match       42
VERB_TIER2_general        25
VERB_TIER1_general        25
Name: count, dtype: int64

Step x Type (HAAD):
kg_step_s2  OK  STOPPED_no_kg_match  STOPPED_no_lonely_adj  \
type                                                         
explicit    78                   42                    129   

kg_step_s2  VERB_TIER1_general  VERB_TIER2_general  
type                                                
explicit                    25                  25  
✅ Saved: KG_Strategy2_HAAD_predictions.csv (299 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Block 12 complete.
   Full dataframes: mabsa_ar, se16_ar, haad_ar (unchanged)
   Run dataframes : mabsa_ar_out, se16_ar_out, haad_ar_out
   Evaluation block will automatically use the run dataframes.


## EVALUATION BLOCK — Strategy 1 vs Strategy 2

**Metric B — KG implicit aspect identification (implicit rows only)**

| Dataset | Gold format | Strategy 1 pred | Strategy 2 pred |
|---|---|---|---|
| M-ABSA Arabic | `ENTITY#ASPECT` (English) | English aspects | Arabic aspects |
| SemEval2016 Arabic | `ENTITY#ASPECT` (English) | English aspects | Arabic aspects |
| HAAD | Arabic aspect strings only | English aspects | Arabic aspects |

**Matching**: aspect-only, partial (head-noun + token-subset), entity ignored.
**Note**: For Strategy 2 on M-ABSA/SE16, Arabic predictions are matched against
English gold aspects via the multilingual normaliser — cross-lingual matching
where Arabic and English equivalent terms may not align textually.


In [ ]:
# ================================================================
# EVALUATION — Arabic KG Inference (Strategy 1 vs Strategy 2)
# ================================================================

import re
from collections import defaultdict

_NULL_VALUES = {
    'none','null','nan','n/a','','[]',"['none']",
    "['null']",'[none]','[null]',"['']",
}

def _is_null_value(v: str) -> bool:
    return str(v).strip().lower() in _NULL_VALUES

_GENERAL_SYN = {
    "misc","miscellaneous","anecdotes/miscellaneous",
    "miscellany","other","general","عام",
}
_IRREG = {
    "rooms":"room","prices":"price","drinks":"drink",
    "services":"service","facilities":"facility",
}
_tok_re_eval = re.compile(r"[a-z0-9؀-ۿ]+")
_DIAC_EVAL   = re.compile(r'[ؗ-ًؚ-ْ]')

def _sing(t: str) -> str:
    t = t.lower()
    if t in _IRREG: return _IRREG[t]
    if re.search(r'[^aeiou]ies$', t): return re.sub(r'ies$','y',t)
    if re.search(r'(sses|ches|shes|xes)$', t): return t[:-2]
    if t.endswith('s') and not t.endswith('ss') and len(t) > 3: return t[:-1]
    return t

def _nf_eval(s: str) -> str:
    s = _DIAC_EVAL.sub('', str(s))
    return " ".join([_sing(x) for x in _tok_re_eval.findall(s.lower())])

def _norm_asp_eval(s: str) -> str:
    n = _nf_eval(s)
    return "general" if n in {_nf_eval(g) for g in _GENERAL_SYN} else n

def _head_tok(s: str) -> str:
    toks = _nf_eval(s).split()
    return toks[-1] if toks else ""

def _same_relaxed_eval(a: str, b: str) -> bool:
    a, b = _norm_asp_eval(a), _norm_asp_eval(b)
    if a == b: return True
    if _head_tok(a) and _head_tok(a) == _head_tok(b): return True
    sa, sb = set(a.split()), set(b.split())
    if sa and sb and (sa.issubset(sb) or sb.issubset(sa)): return True
    return False


def _parse_gold_cat(val, is_haad: bool = False) -> List[Tuple[str, str]]:
    """
    Parse gold category field.
    is_haad=True  → Arabic aspect string only, entity='*'
    is_haad=False → ENTITY#ASPECT English pairs
    """
    if not isinstance(val, list): val = [str(val)]
    pairs = []
    for c in val:
        s = str(c).strip()
        if _is_null_value(s) or s == 'none': continue
        if is_haad:
            asp = _norm_asp_eval(s)
            if asp: pairs.append(('*', asp))
        else:
            if '#' in s:
                e, a = s.split('#', 1)
                pairs.append((_nf_eval(e), _norm_asp_eval(a)))
            else:
                tokens = s.strip().split()
                if len(tokens) >= 2:
                    pairs.append((_nf_eval(tokens[0]),
                                  _norm_asp_eval(' '.join(tokens[1:]))))
                elif tokens:
                    pairs.append(('*', _norm_asp_eval(tokens[0])))
    return pairs


def _parse_preds(val) -> List[Tuple[str, str]]:
    """
    Parse prediction column — handles English or Arabic aspect strings.
    Always returns ('*', aspect) — entity ignored for Arabic evaluation.
    """
    if not isinstance(val, list): val = [str(val)]
    pairs = []
    for x in val:
        s = str(x).strip()
        if _is_null_value(s) or s == 'none': continue
        if '#' in s:
            _, a = s.split('#', 1)
            pairs.append(('*', _norm_asp_eval(a)))
        elif s:
            pairs.append(('*', _norm_asp_eval(s)))
    return pairs


def _max_bip(adj, nl, nr):
    matchR = [-1] * nr
    def dfs(u, seen):
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                if matchR[v] == -1 or dfs(matchR[v], seen):
                    matchR[v] = u; return True
        return False
    for u in range(nl): dfs(u, [False] * nr)
    return [(u, v) for v, u in enumerate(matchR) if u != -1]


def evaluate_kg_arabic(
    df: pd.DataFrame,
    pred_col: str = 'kg_preds',
    gold_col: str = 'category',
    is_haad: bool = False,
) -> Dict:
    """
    Evaluate Arabic KG predictions (partial aspect matching, entity ignored).
    is_haad=True  → gold is Arabic aspect-only
    is_haad=False → gold is ENTITY#ASPECT English pairs
    """
    TP = FP = FN = any_tp = rows = 0
    for _, row in df.iterrows():
        gold = _parse_gold_cat(row.get(gold_col, []), is_haad=is_haad)
        if not gold: continue
        rows += 1
        pred = _parse_preds(row.get(pred_col, []))
        if not pred:
            FN += len(gold); continue
        adj = [
            [j for j, g in enumerate(gold)
             if _same_relaxed_eval(p[1], g[1])]
            for p in pred
        ]
        mc   = len(_max_bip(adj, len(pred), len(gold)))
        TP  += mc; FP += len(pred) - mc; FN += len(gold) - mc
        if mc > 0: any_tp += 1
    p = TP/(TP+FP) if (TP+FP) else 0.0
    r = TP/(TP+FN) if (TP+FN) else 0.0
    f = 2*p*r/(p+r) if (p+r) else 0.0
    return {
        'P': round(p,4), 'R': round(r,4), 'F1': round(f,4),
        'TP': TP, 'FP': FP, 'FN': FN, 'rows': rows,
        'Any-TP@row': round(any_tp/rows, 4) if rows else 0.0,
    }


def _print_result(label, r):
    print(f"    {label:<14}: P={r['P']:.4f}  R={r['R']:.4f}  F1={r['F1']:.4f}  "
          f"(TP={r['TP']} FP={r['FP']} FN={r['FN']})  "
          f"Any-TP@row={r['Any-TP@row']:.4f}  ({r['rows']} rows)")


def _step_dist(df, step_col, impl_only: bool = True):
    sub = df[df['type']=='implicit'] if (impl_only and 'type' in df.columns) else df
    if len(sub) == 0: return
    for step, cnt in sub[step_col].value_counts().items():
        print(f"      {step:<35}: {cnt:4d} ({100*cnt/len(sub):.1f}%)")


# ================================================================
# RUN EVALUATION
# ================================================================
print("=" * 70)
print("ARABIC KG EVALUATION — Metric B (implicit rows, gold=category)")
print("Matching: partial (relaxed) aspect-only, entity ignored")
print("=" * 70)

# is_haad flag per dataset
# Use the run-dataframes from Block 12 (subset in TEST_MODE, full otherwise)
_mabsa = mabsa_ar_out if 'mabsa_ar_out' in dir() else mabsa_ar
_se16  = se16_ar_out  if 'se16_ar_out'  in dir() else se16_ar
_haad  = haad_ar_out  if 'haad_ar_out'  in dir() else haad_ar

DATASETS_EVAL = [
    ("M-ABSA Arabic",    _mabsa, False, 'kg_preds', 'kg_preds_s2'),
    ("SemEval16 Arabic", _se16,  False, 'kg_preds', 'kg_preds_s2'),
    ("HAAD",             _haad,  True,  'kg_preds', 'kg_preds_s2'),
]

for label, df, is_haad, s1_col, s2_col in DATASETS_EVAL:
    if len(df) == 0:
        print(f"\n⚠ {label}: no rows — skipped."); continue

    impl = df[df['type']=='implicit'] if 'type' in df.columns else df
    if len(impl) == 0:
        print(f"\n⚠ {label}: no implicit rows."); continue

    gold_fmt = "Arabic aspect-only" if is_haad else "ENTITY#ASPECT (English)"
    print(f"\n{'─'*65}")
    print(f"  DATASET : {label}  ({len(impl)} implicit rows)")
    print(f"  Gold    : {gold_fmt}")
    print(f"{'─'*65}")

    r1 = r2 = None

    if s1_col in df.columns:
        r1 = evaluate_kg_arabic(impl, pred_col=s1_col,
                                 gold_col='category', is_haad=is_haad)
        _print_result("Strategy 1", r1)
        print(f"    Step dist (S1):")
        _step_dist(df, 'kg_step')
    else:
        print(f"    Strategy 1: '{s1_col}' not found — run Block 9 first.")

    if s2_col in df.columns:
        r2 = evaluate_kg_arabic(impl, pred_col=s2_col,
                                 gold_col='category', is_haad=is_haad)
        _print_result("Strategy 2", r2)
        print(f"    Step dist (S2):")
        _step_dist(df, 'kg_step_s2')
    else:
        print(f"    Strategy 2: '{s2_col}' not found — run Block 12 first.")

    if r1 and r2:
        winner = "Strategy 1" if r1['F1'] >= r2['F1'] else "Strategy 2"
        print(f"    → {winner} wins  (ΔF1 = {abs(r1['F1']-r2['F1']):.4f})")

    if 'domain' in impl.columns:
        domains = sorted(impl['domain'].dropna().unique())
        if len(domains) > 1:
            print(f"\n    Per-domain (implicit rows):")
            print(f"    {'Domain':<15} {'S1 F1':>8} {'S2 F1':>8} {'Winner':>10}")
            print(f"    {'-'*45}")
            for dom in domains:
                sub  = impl[impl['domain'] == dom]
                rd1  = evaluate_kg_arabic(sub, pred_col=s1_col, gold_col='category',
                                          is_haad=is_haad) if s1_col in df.columns else {'F1':0.0}
                rd2  = evaluate_kg_arabic(sub, pred_col=s2_col, gold_col='category',
                                          is_haad=is_haad) if s2_col in df.columns else {'F1':0.0}
                w    = "S1" if rd1['F1'] >= rd2['F1'] else "S2"
                print(f"    {dom:<15} {rd1['F1']:>8.4f} {rd2['F1']:>8.4f} {w:>10}")

# ── Summary table ─────────────────────────────────────────────────
print(f"\n\n{'='*70}")
print("SUMMARY — Strategy 1 vs Strategy 2 (implicit rows F1)")
print(f"{'='*70}")
print(f"  {'Dataset':<22} {'Gold format':<26} {'S1 F1':>8} {'S2 F1':>8} {'Winner':>10}")
print(f"  {'-'*68}")
for label, df, is_haad, s1_col, s2_col in DATASETS_EVAL:
    if len(df) == 0: continue
    impl     = df[df['type']=='implicit'] if 'type' in df.columns else df
    if len(impl) == 0: continue
    gold_fmt = "Arabic-only" if is_haad else "ENTITY#ASPECT EN"
    f1_s1 = evaluate_kg_arabic(impl, pred_col=s1_col, gold_col='category',
                                is_haad=is_haad)['F1'] if s1_col in df.columns else 0.0
    f1_s2 = evaluate_kg_arabic(impl, pred_col=s2_col, gold_col='category',
                                is_haad=is_haad)['F1'] if s2_col in df.columns else 0.0
    winner = "S1" if f1_s1 >= f1_s2 else "S2"
    print(f"  {label:<22} {gold_fmt:<26} {f1_s1:>8.4f} {f1_s2:>8.4f} {winner:>10}")

print("\n✅ Arabic KG evaluation complete.")


ARABIC KG EVALUATION — Metric B (implicit rows, gold=category)
Matching: partial (relaxed) aspect-only, entity ignored

─────────────────────────────────────────────────────────────────
  DATASET : M-ABSA Arabic  (718 implicit rows)
  Gold    : ENTITY#ASPECT (English)
─────────────────────────────────────────────────────────────────
    Strategy 1: 'kg_preds' not found — run Block 9 first.
    Strategy 2    : P=0.5644  R=0.2741  F1=0.3690  (TP=219 FP=169 FN=580)  Any-TP@row=0.3036  (718 rows)
    Step dist (S2):
      STOPPED_no_lonely_adj              :  295 (41.1%)
      OK                                 :  204 (28.4%)
      VERB_TIER1_general                 :   84 (11.7%)
      VERB_TIER2_general                 :   80 (11.1%)
      STOPPED_no_kg_match                :   55 (7.7%)

    Per-domain (implicit rows):
    Domain             S1 F1    S2 F1     Winner
    ---------------------------------------------
    food              0.0000   0.3207         S2
    hotel             